In [1]:
from pathlib import Path
import pandas as pd

project_root = Path(
    r"C:\Users\HP-ZBOOK i7\graphrag_test"
)

pilot1_dir = project_root / "pilot_01" / "inspection"
pilot2_dir = project_root / "pilot_02" / "inspection"

recall_checklist = pd.read_csv(
    pilot1_dir / "14_recall_coverage_checklist.csv"
)

pilot2_entities = pd.read_csv(
    pilot2_dir / "01_entities_all.csv"
)

pilot2_relationships = pd.read_csv(
    pilot2_dir / "04_relationships_all.csv"
)

pilot2_text_units = pd.read_csv(
    pilot2_dir / "09_text_units_all.csv"
)

print("Source facts:", len(recall_checklist))
print("Pilot 2 entities:", len(pilot2_entities))
print("Pilot 2 relationships:", len(pilot2_relationships))
print("Pilot 2 text units:", len(pilot2_text_units))

print("\nChecklist columns:")
print(recall_checklist.columns.tolist())

Source facts: 30
Pilot 2 entities: 539
Pilot 2 relationships: 643
Pilot 2 text units: 17

Checklist columns:
['fact_id', 'category', 'source_section', 'source_fact', 'expected_entities', 'expected_relationship', 'text_unit_capture', 'entity_capture', 'relationship_capture', 'community_report_capture', 'overall_classification', 'audit_notes', 'text_unit_evidence']


In [2]:
pilot1_result_columns = [
    "text_unit_capture",
    "text_unit_evidence",
    "entity_capture",
    "relationship_capture",
    "overall_classification",
    "audit_notes",
]

for column in pilot1_result_columns:
    if column in recall_checklist.columns:
        recall_checklist.rename(
            columns={column: f"pilot1_{column}"},
            inplace=True,
        )

pilot2_columns = [
    "pilot2_text_unit_capture",
    "pilot2_text_unit_evidence",
    "pilot2_entity_capture",
    "pilot2_entity_evidence",
    "pilot2_relationship_capture",
    "pilot2_relationship_evidence",
    "pilot2_overall_classification",
    "pilot2_audit_notes",
]

for column in pilot2_columns:
    recall_checklist[column] = ""

print(recall_checklist.columns.tolist())

recall_checklist[
    [
        "fact_id",
        "category",
        "source_fact",
        "expected_entities",
        "expected_relationship",
    ]
].head()

['fact_id', 'category', 'source_section', 'source_fact', 'expected_entities', 'expected_relationship', 'pilot1_text_unit_capture', 'pilot1_entity_capture', 'pilot1_relationship_capture', 'community_report_capture', 'pilot1_overall_classification', 'pilot1_audit_notes', 'pilot1_text_unit_evidence', 'pilot2_text_unit_capture', 'pilot2_text_unit_evidence', 'pilot2_entity_capture', 'pilot2_entity_evidence', 'pilot2_relationship_capture', 'pilot2_relationship_evidence', 'pilot2_overall_classification', 'pilot2_audit_notes']


,fact_id,category,source_fact,expected_entities,expected_relationship
0,P01,POLICY,The Renewable Expansion Act establishes that t...,Austria; Renewable Expansion Act; Renewable El...,Austria HAS_TARGET 100% nationally balanced re...
1,P02,POLICY,The Renewable Expansion Act provides for at le...,Renewable Expansion Act; Photovoltaics; 11 TWh...,Renewable Expansion Act HAS_TARGET 11 TWh of a...
2,P03,POLICY,RED III raises the EU renewable-energy share t...,RED III; European Union; 42.5%; 45%; 2030,RED III HAS_TARGET EU renewable-energy share b...
3,P04,POLICY,Austria introduced a zero-percent VAT measure ...,VAT Exemption; PV System; 35 kWp,VAT Exemption SUPPORTS PV System up to 35 kWp
4,P05,POLICY,The proposed Electricity Industry Act includes...,Electricity Industry Act; Grid Connection; Gri...,Electricity Industry Act SUPPORTS simplified a...


In [3]:
recall_checklist["pilot2_text_unit_capture"] = "YES"

if "pilot1_text_unit_evidence" in recall_checklist.columns:
    recall_checklist["pilot2_text_unit_evidence"] = (
        recall_checklist["pilot1_text_unit_evidence"]
    )

print("Pilot 2 text-unit capture:")
print(
    recall_checklist[
        "pilot2_text_unit_capture"
    ].value_counts()
)

Pilot 2 text-unit capture:
pilot2_text_unit_capture
YES    30
Name: count, dtype: int64


In [4]:
def clean_text(series):
    return (
        series
        .fillna("")
        .astype(str)
        .str.upper()
    )

pilot2_entities["search_text"] = (
    clean_text(pilot2_entities["title"])
    + " "
    + clean_text(pilot2_entities["description"])
)

pilot2_relationships["search_text"] = (
    clean_text(pilot2_relationships["source"])
    + " "
    + clean_text(pilot2_relationships["target"])
    + " "
    + clean_text(pilot2_relationships["description"])
)

print("Searchable entity and relationship text created.")

Searchable entity and relationship text created.


In [5]:
policy_search_terms = {
    "P01": [
        "ERNEUERBAREN-AUSBAU-GESETZ",
        "100%",
        "2030",
    ],
    "P02": [
        "ERNEUERBAREN-AUSBAU-GESETZ",
        "11 TWH",
        "2030",
    ],
    "P03": [
        "RED III",
        "42.5%",
        "45%",
        "2030",
    ],
    "P04": [
        "UMSATZSTEUER",
        "35 KWP",
        "PV-ANLAGE",
    ],
    "P05": [
        "ELEKTRIZITÄTSWIRTSCHAFTSGESETZ",
        "NETZANSCHLUSS",
    ],
    "P06": [
        "ERNEUERBAREN-AUSBAU-BESCHLEUNIGUNGSGESETZ",
        "EABG",
    ],
    "P07": [
        "BAURECHT",
        "RAUMORDNUNG",
        "NATURSCHUTZ",
        "BUNDESLÄNDER",
    ],
    "P08": [
        "BUND-LÄNDER-DIALOG",
        "ERNEUERBARE ENERGIE",
    ],
    "P09": [
        "WOHNUNGSEIGENTUMSGESETZ",
        "NOVELLE 2022",
    ],
    "P10": [
        "PHOTOVOLTAIK-STRATEGIE",
        "LEBENDES DOKUMENT",
        "ANPASSUNG",
    ],
}

In [6]:
def find_candidates(dataframe, terms, maximum=15):
    pattern = "|".join(
        pd.Series(terms)
        .str.replace(r"([.^$*+?{}\[\]\\|()])", r"\\\1", regex=True)
    )

    result = dataframe[
        dataframe["search_text"].str.contains(
            pattern,
            regex=True,
            na=False,
        )
    ].copy()

    if "degree" in result.columns:
        result = result.sort_values(
            ["degree", "frequency"],
            ascending=False,
        )

    return result.head(maximum)


for fact_id, terms in policy_search_terms.items():
    source_fact = recall_checklist.loc[
        recall_checklist["fact_id"] == fact_id,
        "source_fact",
    ].iloc[0]

    print("\n" + "=" * 100)
    print(f"{fact_id}: {source_fact}")
    print("Search terms:", terms)

    entity_candidates = find_candidates(
        pilot2_entities,
        terms,
        maximum=12,
    )

    relationship_candidates = find_candidates(
        pilot2_relationships,
        terms,
        maximum=15,
    )

    print("\nENTITY CANDIDATES:")
    display(
        entity_candidates[
            [
                "title",
                "type",
                "frequency",
                "degree",
                "description",
            ]
        ]
    )

    print("\nRELATIONSHIP CANDIDATES:")
    display(
        relationship_candidates[
            [
                "source",
                "target",
                "description",
                "weight",
            ]
        ]
    )


P01: The Renewable Expansion Act establishes that total electricity consumption should be covered nationally and on balance by 100% renewable energy from 2030.
Search terms: ['ERNEUERBAREN-AUSBAU-GESETZ', '100%', '2030']

ENTITY CANDIDATES:


,title,type,frequency,degree,description
10,ERNEUERBAREN-AUSBAU-GESETZ (EAG),POLICY,5,10,"The Erneuerbaren-Ausbau-Gesetz (EAG), also kno..."
8,100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,TARGET,1,5,Target: nationally balanced renewable electric...
173,ERNEUERBAREN-AUSBAU-GESETZ,POLICY,2,4,The ERNEUERBAREN-AUSBAU-GESETZ (Renewable Expa...
44,PV-AUSBAU VON MINDESTENS 11 TWH BIS 2030,TARGET,1,4,Target: PV electricity generation increase; Ta...
157,PV-STROMERZEUGUNG ERFORDERLICH — 21 TWH (ÖSTER...,TARGET,1,4,Target: required photovoltaic electricity gene...
227,100% NATIONAL BILANZIELL ERNEUERBARER GESAMTST...,TARGET,1,4,Target: complete national coverage of total el...
263,PV-AUSBAUZIELE 2030,TARGET,1,4,Target: PV expansion for 2030; Target value: n...
264,PV-AUSBAUZIELE 2040,TARGET,1,3,Target: PV expansion for 2040; Target value: n...
158,PV-ZUWACHS GEGENÜBER 2020 ERFORDERLICH — 19 TW...,TARGET,1,2,Target: required PV expansion increase compare...
159,EAG GENANNTE ZUBAUMENGE PV BIS 2030 GEGENÜBER ...,TARGET,1,2,Target: PV expansion quantity stated in EAG co...



RELATIONSHIP CANDIDATES:


,source,target,description,weight
4,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLI...,9.0
5,PHOTOVOLTAIK,100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLI...,10.0
7,ÖSTERREICH,100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,[RELATION_TYPE=HAS_TARGET] [MODALITY=EXPLICIT_...,10.0
8,ERNEUERBAREN-AUSBAU-GESETZ (EAG),100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,[RELATION_TYPE=SETS_TARGET] [MODALITY=EXPLICIT...,10.0
15,ERNEUERBARE ENERGIEN,100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLI...,10.0
36,PHOTOVOLTAIK-STRATEGIE,PV-AUSBAU VON MINDESTENS 11 TWH BIS 2030,[RELATION_TYPE=SETS_TARGET] [MODALITY=EXPLICIT...,10.0
37,ERNEUERBAREN-AUSBAU-GESETZ (EAG),PV-AUSBAU VON MINDESTENS 11 TWH BIS 2030,[RELATION_TYPE=SETS_TARGET] [MODALITY=EXPLICIT...,10.0
38,ERNEUERBAREN-AUSBAU-GESETZ (EAG),100% NATIONAL BILANZIERTER ERNEUERBARER STROMV...,[RELATION_TYPE=SETS_TARGET] [MODALITY=EXPLICIT...,10.0
39,ÖSTERREICH,PV-AUSBAU VON MINDESTENS 11 TWH BIS 2030,[RELATION_TYPE=HAS_TARGET] [MODALITY=EXPLICIT_...,10.0
40,PHOTOVOLTAIK,PV-AUSBAU VON MINDESTENS 11 TWH BIS 2030,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLI...,9.0



P02: The Renewable Expansion Act provides for at least 11 TWh of additional photovoltaic generation by 2030 compared with 2020.
Search terms: ['ERNEUERBAREN-AUSBAU-GESETZ', '11 TWH', '2030']

ENTITY CANDIDATES:


...

,source,target,description,weight
215,WOHNUNGSEIGENTUMSGESETZ (NOVELLE 2022),PHOTOVOLTAIKANLAGEN,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,10.0
243,WOHNUNGSEIGENTUMSGESETZ-NOVELLE 2022,FOTOVOLTAIKANLAGE,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,10.0
244,WOHNUNGSEIGENTUMSGESETZ-NOVELLE 2022,REIHENHÄUSER UND EINZELGEBÄUDE,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,10.0



P10: The Austrian Photovoltaic Strategy is intended to be a living document that is adjusted when conditions change.
Search terms: ['PHOTOVOLTAIK-STRATEGIE', 'LEBENDES DOKUMENT', 'ANPASSUNG']

ENTITY CANDIDATES:


,title,type,frequency,degree,description
6,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,POLICY,10,45,"The ""ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE"" (..."
31,PHOTOVOLTAIK-STRATEGIE,POLICY,1,21,Policy type: strategy; Jurisdiction: Austria; ...
269,ANPASSUNG DER ZONIERUNGEN AN DEN AUSBAUBEDARF ...,POLICY,1,1,Policy type: strategic spatial planning action...
270,REGELMÄSSIGE ANPASSUNG DER FLÄCHENPOTENTIALE I...,POLICY,1,1,Policy type: planning requirement; Jurisdictio...



RELATIONSHIP CANDIDATES:


,source,target,description,weight
0,"BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENE...",ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,[RELATION_TYPE=RESPONSIBLE_FOR] [MODALITY=EXPL...,10.0
2,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,PHOTOVOLTAIK,The Austrian Photovoltaic Strategy (ÖSTERREICH...,30.0
3,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,KLIMANEUTRALITÄT IN ÖSTERREICH BIS 2040,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLI...,9.0
4,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLI...,9.0
24,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,RAHMENBEDINGUNGEN FÜR DEN PV-AUSBAU,[RELATION_TYPE=ASSOCIATED_WITH] [MODALITY=EXPL...,10.0
25,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,ZIELSETZUNGEN FÜR DEN PV-AUSBAU,[RELATION_TYPE=SETS_TARGET] [MODALITY=EXPLICIT...,10.0
26,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,MASSNAHMEN FÜR DEN PV-AUSBAU,[RELATION_TYPE=PROPOSES] [MODALITY=EXPLICIT_FA...,10.0
36,PHOTOVOLTAIK-STRATEGIE,PV-AUSBAU VON MINDESTENS 11 TWH BIS 2030,[RELATION_TYPE=SETS_TARGET] [MODALITY=EXPLICIT...,10.0
42,PHOTOVOLTAIK-STRATEGIE,RECHTLICHER RAHMEN,[RELATION_TYPE=PROPOSES] [MODALITY=EXPLICIT_FA...,8.0
43,PHOTOVOLTAIK-STRATEGIE,TECHNISCHE UND SYSTEMISCHE FRAGEN,[RELATION_TYPE=PROPOSES] [MODALITY=EXPLICIT_FA...,8.0


In [7]:
policy_recall_results = {
    "P01": {
        "entity": "FULL",
        "entity_evidence": (
            "ERNEUERBAREN-AUSBAU-GESETZ (EAG) and the TARGET entity "
            "100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BIS 2030."
        ),
        "relationship": "FULL",
        "relationship_evidence": (
            "EAG SETS_TARGET 100% nationally balanced renewable "
            "electricity consumption by 2030; Austria also HAS_TARGET."
        ),
        "overall": "FULLY_STRUCTURED",
        "notes": (
            "Policy, target value, geographic scope and 2030 deadline "
            "are explicitly represented."
        ),
    },

    "P02": {
        "entity": "FULL",
        "entity_evidence": (
            "EAG, PHOTOVOLTAIK and PV-AUSBAU VON MINDESTENS "
            "11 TWH BIS 2030 are extracted."
        ),
        "relationship": "FULL",
        "relationship_evidence": (
            "EAG SETS_TARGET PV expansion of at least 11 TWh by "
            "2030, with an explicit connection to photovoltaics."
        ),
        "overall": "FULLY_STRUCTURED",
        "notes": (
            "The legal instrument, technology, numerical quantity, "
            "deadline and comparison context are represented."
        ),
    },

    "P03": {
        "entity": "FULL",
        "entity_evidence": (
            "EU-ERNEUERBAREN-RICHTLINIE (RED III), the European "
            "Union and the renewable-energy-share target are extracted."
        ),
        "relationship": "FULL",
        "relationship_evidence": (
            "RED III SETS_TARGET the renewable-energy share, including "
            "the 42.5%, 45% ambition and 2030 context."
        ),
        "overall": "FULLY_STRUCTURED",
        "notes": (
            "Pilot 2 represents this numerical EU policy target as a "
            "policy-to-target relationship rather than a generic event."
        ),
    },

    "P04": {
        "entity": "FULL",
        "entity_evidence": (
            "UMSATZSTEUERBEFREIUNG FÜR PV-ANLAGEN BIS 35 KWP is "
            "extracted as SUPPORT_SCHEME, together with PV-ANLAGE."
        ),
        "relationship": "PARTIAL",
        "relationship_evidence": (
            "The zero-percent VAT support scheme and eligible PV "
            "installation threshold are represented, but the displayed "
            "relationship candidates do not clearly preserve the complete "
            "support-scheme-to-35-kWp relationship in one edge."
        ),
        "overall": "PARTIALLY_STRUCTURED",
        "notes": (
            "The important entities and threshold are captured. The exact "
            "support relationship should still be verified before treating "
            "the fact as fully structured."
        ),
    },

    "P05": {
        "entity": "FULL",
        "entity_evidence": (
            "ELEKTRIZITÄTSWIRTSCHAFTSGESETZ (ELWG), NETZANSCHLUSS, "
            "NETZANSCHLUSSVERFAHREN, VERTEILERNETZ and related "
            "infrastructure concepts are extracted."
        ),
        "relationship": "PARTIAL",
        "relationship_evidence": (
            "Multiple relationships connect the ELWG to distribution-grid "
            "regulation, grid access, digital infrastructure and PV systems. "
            "The complete compound claim concerning simplified access for "
            "small systems, transparency, capacity data and development "
            "plans is distributed across several edges."
        ),
        "overall": "PARTIALLY_STRUCTURED",
        "notes": (
            "Pilot 2 captures the policy and its principal components, but "
            "not the complete compound fact as one precise relationship."
        ),
    },

    "P06": {
        "entity": "FULL",
        "entity_evidence": (
            "ERNEUERBAREN-AUSBAU-BESCHLEUNIGUNGSGESETZ appears as "
            "POLICY, including EABG and E-ABG variants."
        ),
        "relationship": "FULL",
        "relationship_evidence": (
            "The act SUPPORTS photovoltaic installations, simplified "
            "procedures, procedural exemptions and PV uses in infrastructure."
        ),
        "overall": "FULLY_STRUCTURED",
        "notes": (
            "The planned act and its intended acceleration mechanisms are "
            "explicitly represented, although alias resolution is still needed."
        ),
    },

    "P07": {
        "entity": "PARTIAL",
        "entity_evidence": (
            "BUNDESLÄNDER and NATURSCHUTZ are extracted, together with "
            "state-level targets and land-related concepts. Building law "
            "and spatial planning are not clearly represented as corresponding "
            "controlled entities in the displayed evidence."
        ),
        "relationship": "PARTIAL",
        "relationship_evidence": (
            "Federal states are linked to land availability, permitting "
            "simplification and target setting, while nature conservation "
            "constrains PV. The complete assignment of building law, spatial "
            "planning and nature conservation to federal-state responsibility "
            "is not represented in one complete relationship structure."
        ),
        "overall": "PARTIALLY_STRUCTURED",
        "notes": (
            "The central actor and part of its responsibilities are captured, "
            "but the full responsibility statement is fragmented."
        ),
    },

    "P08": {
        "entity": "FULL",
        "entity_evidence": (
            "BUND-LÄNDER-DIALOG ERNEUERBARE ENERGIE is extracted, "
            "together with BUND and BUNDESLÄNDER."
        ),
        "relationship": "PARTIAL",
        "relationship_evidence": (
            "The dialogue is explicitly associated with the federal "
            "government and federal states and linked to renewable targets. "
            "Coordination of land availability, processes and support "
            "strategies is not fully expressed in the displayed edges."
        ),
        "overall": "PARTIALLY_STRUCTURED",
        "notes": (
            "The coordinating mechanism and participating government levels "
            "are captured, but the complete scope of coordination is incomplete."
        ),
    },

    "P09": {
        "entity": "FULL",
        "entity_evidence": (
            "WOHNUNGSEIGENTUMSGESETZ-NOVELLE 2022 and its spelling "
            "variant are extracted as POLICY entities."
        ),
        "relationship": "FULL",
        "relationship_evidence": (
            "The amendment SUPPORTS photovoltaic installations and is "
            "linked explicitly to terraced houses and individual buildings."
        ),
        "overall": "FULLY_STRUCTURED",
        "notes": (
            "The policy amendment, relevant building types and support for "
            "PV construction are explicitly represented. Duplicate policy "
            "names require later entity resolution."
        ),
    },

    "P10": {
        "entity": "PARTIAL",
        "entity_evidence": (
            "The Austrian Photovoltaic Strategy is extracted as POLICY, "
            "but the living-document or adaptive-policy characteristic is "
            "not clearly represented as a separate entity."
        ),
        "relationship": "MISSING",
        "relationship_evidence": (
            "The displayed strategy relationships cover targets and action "
            "fields, but no relationship explicitly states that the strategy "
            "is revised or evolves when conditions change."
        ),
        "overall": "ENTITY_ONLY",
        "notes": (
            "The strategy itself is strongly represented, but its adaptive "
            "or living-document property was not converted into a structured "
            "relationship."
        ),
    },
}

In [8]:
for fact_id, result in policy_recall_results.items():
    mask = recall_checklist["fact_id"] == fact_id

    recall_checklist.loc[
        mask, "pilot2_entity_capture"
    ] = result["entity"]

    recall_checklist.loc[
        mask, "pilot2_entity_evidence"
    ] = result["entity_evidence"]

    recall_checklist.loc[
        mask, "pilot2_relationship_capture"
    ] = result["relationship"]

    recall_checklist.loc[
        mask, "pilot2_relationship_evidence"
    ] = result["relationship_evidence"]

    recall_checklist.loc[
        mask, "pilot2_overall_classification"
    ] = result["overall"]

    recall_checklist.loc[
        mask, "pilot2_audit_notes"
    ] = result["notes"]


In [9]:
policy_results = recall_checklist[
    recall_checklist["category"] == "POLICY"
]

policy_results[
    [
        "fact_id",
        "source_fact",
        "pilot2_entity_capture",
        "pilot2_relationship_capture",
        "pilot2_overall_classification",
        "pilot2_audit_notes",
    ]
]

,fact_id,source_fact,pilot2_entity_capture,pilot2_relationship_capture,pilot2_overall_classification,pilot2_audit_notes
0,P01,The Renewable Expansion Act establishes that t...,FULL,FULL,FULLY_STRUCTURED,"Policy, target value, geographic scope and 203..."
1,P02,The Renewable Expansion Act provides for at le...,FULL,FULL,FULLY_STRUCTURED,"The legal instrument, technology, numerical qu..."
2,P03,RED III raises the EU renewable-energy share t...,FULL,FULL,FULLY_STRUCTURED,Pilot 2 represents this numerical EU policy ta...
3,P04,Austria introduced a zero-percent VAT measure ...,FULL,PARTIAL,PARTIALLY_STRUCTURED,The important entities and threshold are captu...
4,P05,The proposed Electricity Industry Act includes...,FULL,PARTIAL,PARTIALLY_STRUCTURED,Pilot 2 captures the policy and its principal ...
5,P06,The planned Renewable Expansion Acceleration A...,FULL,FULL,FULLY_STRUCTURED,The planned act and its intended acceleration ...
6,P07,"Building law, spatial planning and nature cons...",PARTIAL,PARTIAL,PARTIALLY_STRUCTURED,The central actor and part of its responsibili...
7,P08,The Federal-State Dialogue on Renewable Energy...,FULL,PARTIAL,PARTIALLY_STRUCTURED,The coordinating mechanism and participating g...
8,P09,The 2022 amendment to the Condominium Act made...,FULL,FULL,FULLY_STRUCTURED,"The policy amendment, relevant building types ..."
9,P10,The Austrian Photovoltaic Strategy is intended...,PARTIAL,MISSING,ENTITY_ONLY,"The strategy itself is strongly represented, b..."


In [10]:
print("Policy entity capture:")
print(
    policy_results[
        "pilot2_entity_capture"
    ].value_counts()
)

print("\nPolicy relationship capture:")
print(
    policy_results[
        "pilot2_relationship_capture"
    ].value_counts()
)

print("\nPolicy overall classification:")
print(
    policy_results[
        "pilot2_overall_classification"
    ].value_counts()
)

Policy entity capture:
pilot2_entity_capture
FULL       8
PARTIAL    2
Name: count, dtype: int64

Policy relationship capture:
pilot2_relationship_capture
FULL       5
PARTIAL    4
MISSING    1
Name: count, dtype: int64

Policy overall classification:
pilot2_overall_classification
FULLY_STRUCTURED        5
PARTIALLY_STRUCTURED    4
ENTITY_ONLY             1
Name: count, dtype: int64


In [14]:
market_search_terms = {
    "M01": ["6.3 TWH", "6,3 TWH", "2023"],
    "M02": ["2.5 GW", "2,5 GW", "2023"],
    "M03": ["1 GW", "2022"],
    "M04": ["41 TWH", "2040", "ÖNIP", "NIP"],
    "M05": ["15 TWH", "2040", "GEBÄUDE"],
    "M06": ["2 GW", "JÄHRLICH", "2040"],
    "M07": ["21 TWH", "2030", "TRANSITION 2040"],
    "M08": ["19 TWH", "2020", "2030"],
    "M09": ["600 MIO", "600 MILLION", "2023"],
    "M10": ["10%", "2%", "2023"],
}

for fact_id, terms in market_search_terms.items():

    fact = recall_checklist.loc[
        recall_checklist["fact_id"] == fact_id,
        "source_fact"
    ].iloc[0]

    print("\n" + "=" * 90)
    print(f"{fact_id}: {fact}")
    print("Search terms:", terms)

    # Search entities and relationships separately
    entity_candidates = find_candidates(
        pilot2_entities,
        terms
    )

    relationship_candidates = find_candidates(
        pilot2_relationships,
        terms
    )

    print("\nENTITY CANDIDATES:")
    display(
        entity_candidates[
            [
                "title",
                "type",
                "frequency",
                "degree",
                "description",
            ]
        ].head(15)
    )

    print("\nRELATIONSHIP CANDIDATES:")
    display(
        relationship_candidates[
            [
                "source",
                "target",
                "description",
                "weight",
            ]
        ].head(20)
    )


M01: Austria reached 6.3 TWh of PV generation in 2023.
Search terms: ['6.3 TWH', '6,3 TWH', '2023']

ENTITY CANDIDATES:


,title,type,frequency,degree,description
32,KLIMA- UND ENERGIEFONDS,ORGANIZATION,5,8,The KLIMA- UND ENERGIEFONDS (Climate and Energ...
34,E-CONTROL,ORGANIZATION,5,8,"E-CONTROL, also referred to as E-Control, is t..."
47,"PV-GENERATION — 6.3 TWH (ÖSTERREICH, 2023)",MARKET_METRIC,1,3,Metric: PV electricity generation; Value: 6.3;...
48,"PV-FÖRDERUNG — 600 MIO. EUR (ÖSTERREICH, 2023)",MARKET_METRIC,1,3,Metric: PV support/funding; Value: 600; Unit: ...
13,"JÄHRLICHER PV-ZUBAU — 2,5 GW (ÖSTERREICH, 2023)",MARKET_METRIC,1,2,Metric: annual PV capacity expansion; Value: 2...
79,ANTEIL FOSSILE ENERGIETRÄGER AM ENERGIEEINSATZ...,MARKET_METRIC,1,2,Metric: share of fossil energy carriers in tot...
81,"PV-GENERIERTE ENERGIE — 6,3 TWH (ÖSTERREICH, 2...",MARKET_METRIC,1,2,Metric: PV-generated energy; Value: 6.3; Unit:...
154,PV-ANTEIL AM STROMENDVERBRAUCH — CA. 10% (ÖSTE...,MARKET_METRIC,1,2,Metric: share of photovoltaics in final electr...
155,PV-ANTEIL AM ENERGIEBEDARF — CA. 2% (ÖSTERREIC...,MARKET_METRIC,1,2,Metric: share of photovoltaics in energy deman...
163,REKORDMARKEN PV-NEUINSTALLATION — 1 GW (2022);...,MARKET_METRIC,1,2,Metric: PV new installations; Value: 1 GW in 2...



RELATIONSHIP CANDIDATES:


,source,target,description,weight
13,ÖSTERREICH,"JÄHRLICHER PV-ZUBAU — 2,5 GW (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
14,PHOTOVOLTAIK,"JÄHRLICHER PV-ZUBAU — 2,5 GW (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
50,ÖSTERREICH,"PV-GENERATION — 6.3 TWH (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
51,PV-ANLAGEN,"PV-GENERATION — 6.3 TWH (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
52,PHOTOVOLTAIK,"PV-GENERATION — 6.3 TWH (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
53,ÖSTERREICH,"PV-FÖRDERUNG — 600 MIO. EUR (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
54,KLIMA- UND ENERGIEFONDS,"PV-FÖRDERUNG — 600 MIO. EUR (ÖSTERREICH, 2023)",[RELATION_TYPE=FUNDS] [MODALITY=EXPLICIT_FACT]...,9.0
55,ERNEUERBAREN-AUSBAU-GESETZ (EAG),"PV-FÖRDERUNG — 600 MIO. EUR (ÖSTERREICH, 2023)",[RELATION_TYPE=FUNDS] [MODALITY=EXPLICIT_FACT]...,9.0
79,FOSSILE ENERGIETRÄGER,ANTEIL FOSSILE ENERGIETRÄGER AM ENERGIEEINSATZ...,ANTEIL FOSSILE ENERGIETRÄGER AM ENERGIEEINSATZ...,20.0
80,ÖSTERREICH,ANTEIL FOSSILE ENERGIETRÄGER AM ENERGIEEINSATZ...,[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0



M02: Approximately 2.5 GW of new photovoltaic capacity was installed in Austria in 2023.
Search terms: ['2.5 GW', '2,5 GW', '2023']

ENTITY CANDIDATES:


...

,source,target,description,weight
13,ÖSTERREICH,"JÄHRLICHER PV-ZUBAU — 2,5 GW (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
14,PHOTOVOLTAIK,"JÄHRLICHER PV-ZUBAU — 2,5 GW (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
50,ÖSTERREICH,"PV-GENERATION — 6.3 TWH (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
51,PV-ANLAGEN,"PV-GENERATION — 6.3 TWH (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
52,PHOTOVOLTAIK,"PV-GENERATION — 6.3 TWH (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
53,ÖSTERREICH,"PV-FÖRDERUNG — 600 MIO. EUR (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
54,KLIMA- UND ENERGIEFONDS,"PV-FÖRDERUNG — 600 MIO. EUR (ÖSTERREICH, 2023)",[RELATION_TYPE=FUNDS] [MODALITY=EXPLICIT_FACT]...,9.0
55,ERNEUERBAREN-AUSBAU-GESETZ (EAG),"PV-FÖRDERUNG — 600 MIO. EUR (ÖSTERREICH, 2023)",[RELATION_TYPE=FUNDS] [MODALITY=EXPLICIT_FACT]...,9.0
79,FOSSILE ENERGIETRÄGER,ANTEIL FOSSILE ENERGIETRÄGER AM ENERGIEEINSATZ...,ANTEIL FOSSILE ENERGIETRÄGER AM ENERGIEEINSATZ...,20.0
80,ÖSTERREICH,ANTEIL FOSSILE ENERGIETRÄGER AM ENERGIEEINSATZ...,[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0



M10: At the end of 2023, PV supplied almost 10% of Austria's final electricity consumption and approximately 2% of total energy demand.
Search terms: ['10%', '2%', '2023']

ENTITY CANDIDATES:


,title,type,frequency,degree,description
32,KLIMA- UND ENERGIEFONDS,ORGANIZATION,5,8,The KLIMA- UND ENERGIEFONDS (Climate and Energ...
34,E-CONTROL,ORGANIZATION,5,8,"E-CONTROL, also referred to as E-Control, is t..."
47,"PV-GENERATION — 6.3 TWH (ÖSTERREICH, 2023)",MARKET_METRIC,1,3,Metric: PV electricity generation; Value: 6.3;...
48,"PV-FÖRDERUNG — 600 MIO. EUR (ÖSTERREICH, 2023)",MARKET_METRIC,1,3,Metric: PV support/funding; Value: 600; Unit: ...
13,"JÄHRLICHER PV-ZUBAU — 2,5 GW (ÖSTERREICH, 2023)",MARKET_METRIC,1,2,Metric: annual PV capacity expansion; Value: 2...
79,ANTEIL FOSSILE ENERGIETRÄGER AM ENERGIEEINSATZ...,MARKET_METRIC,1,2,Metric: share of fossil energy carriers in tot...
81,"PV-GENERIERTE ENERGIE — 6,3 TWH (ÖSTERREICH, 2...",MARKET_METRIC,1,2,Metric: PV-generated energy; Value: 6.3; Unit:...
154,PV-ANTEIL AM STROMENDVERBRAUCH — CA. 10% (ÖSTE...,MARKET_METRIC,1,2,Metric: share of photovoltaics in final electr...
155,PV-ANTEIL AM ENERGIEBEDARF — CA. 2% (ÖSTERREIC...,MARKET_METRIC,1,2,Metric: share of photovoltaics in energy deman...
163,REKORDMARKEN PV-NEUINSTALLATION — 1 GW (2022);...,MARKET_METRIC,1,2,Metric: PV new installations; Value: 1 GW in 2...



RELATIONSHIP CANDIDATES:


,source,target,description,weight
13,ÖSTERREICH,"JÄHRLICHER PV-ZUBAU — 2,5 GW (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
14,PHOTOVOLTAIK,"JÄHRLICHER PV-ZUBAU — 2,5 GW (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
50,ÖSTERREICH,"PV-GENERATION — 6.3 TWH (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
51,PV-ANLAGEN,"PV-GENERATION — 6.3 TWH (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
52,PHOTOVOLTAIK,"PV-GENERATION — 6.3 TWH (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
53,ÖSTERREICH,"PV-FÖRDERUNG — 600 MIO. EUR (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
54,KLIMA- UND ENERGIEFONDS,"PV-FÖRDERUNG — 600 MIO. EUR (ÖSTERREICH, 2023)",[RELATION_TYPE=FUNDS] [MODALITY=EXPLICIT_FACT]...,9.0
55,ERNEUERBAREN-AUSBAU-GESETZ (EAG),"PV-FÖRDERUNG — 600 MIO. EUR (ÖSTERREICH, 2023)",[RELATION_TYPE=FUNDS] [MODALITY=EXPLICIT_FACT]...,9.0
79,FOSSILE ENERGIETRÄGER,ANTEIL FOSSILE ENERGIETRÄGER AM ENERGIEEINSATZ...,ANTEIL FOSSILE ENERGIETRÄGER AM ENERGIEEINSATZ...,20.0
80,ÖSTERREICH,ANTEIL FOSSILE ENERGIETRÄGER AM ENERGIEEINSATZ...,[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0


In [15]:
fact_id = "M08"
terms = ["19 TWH"]

fact = recall_checklist.loc[
    recall_checklist["fact_id"] == fact_id,
    "source_fact"
].iloc[0]

print(f"{fact_id}: {fact}")

entity_candidates = find_candidates(
    pilot2_entities,
    terms
)

relationship_candidates = find_candidates(
    pilot2_relationships,
    terms
)

print("\nENTITY CANDIDATES:")
display(
    entity_candidates[
        ["title", "type", "frequency", "degree", "description"]
    ].head(30)
)

print("\nRELATIONSHIP CANDIDATES:")
display(
    relationship_candidates[
        ["source", "target", "description", "weight"]
    ].head(30)
)

M08: The 21 TWh requirement for 2030 represents an increase of 19 TWh compared with 2020.

ENTITY CANDIDATES:


,title,type,frequency,degree,description
158,PV-ZUWACHS GEGENÜBER 2020 ERFORDERLICH — 19 TW...,TARGET,1,2,Target: required PV expansion increase compare...



RELATIONSHIP CANDIDATES:


,source,target,description,weight
177,PV-STROMERZEUGUNG ERFORDERLICH — 21 TWH (ÖSTER...,PV-ZUWACHS GEGENÜBER 2020 ERFORDERLICH — 19 TW...,[RELATION_TYPE=PART_OF] [MODALITY=EXPLICIT_FAC...,9.0
178,PV-ZUWACHS GEGENÜBER 2020 ERFORDERLICH — 19 TW...,EAG GENANNTE ZUBAUMENGE PV BIS 2030 GEGENÜBER ...,[RELATION_TYPE=ASSOCIATED_WITH] [MODALITY=EXPL...,9.0


In [16]:
fact_id = "M05"
terms = ["15 TWH"]

fact = recall_checklist.loc[
    recall_checklist["fact_id"] == fact_id,
    "source_fact"
].iloc[0]

entity_candidates = find_candidates(
    pilot2_entities,
    terms
)

relationship_candidates = find_candidates(
    pilot2_relationships,
    terms
)

print(f"{fact_id}: {fact}")

print("\nENTITY CANDIDATES:")
display(
    entity_candidates[
        ["title", "type", "frequency", "degree", "description"]
    ]
)

print("\nRELATIONSHIP CANDIDATES:")
display(
    relationship_candidates[
        ["source", "target", "description", "weight"]
    ]
)

M05: At least approximately 15 TWh of the 2040 PV potential is considered achievable on existing and new buildings.

ENTITY CANDIDATES:


,title,type,frequency,degree,description
153,POTENZIAL PV AUF GEBÄUDEN — 15 TWH BY 2040,TARGET,1,2,Target: PV generation potential on existing an...



RELATIONSHIP CANDIDATES:


,source,target,description,weight
168,AUSBBAUPOTENTIAL PHOTOVOLTAIK — 41 TWH BY 2040,POTENZIAL PV AUF GEBÄUDEN — 15 TWH BY 2040,[RELATION_TYPE=PART_OF] [MODALITY=EXPLICIT_FAC...,10.0
172,ÖSTERREICH,POTENZIAL PV AUF GEBÄUDEN — 15 TWH BY 2040,[RELATION_TYPE=HAS_TARGET] [MODALITY=EXPLICIT_...,10.0


In [17]:
fact_id = "M06"
terms = ["2 GW", "JÄHRLICH", "2040"]

fact = recall_checklist.loc[
    recall_checklist["fact_id"] == fact_id,
    "source_fact"
].iloc[0]

print(f"{fact_id}: {fact}")

entity_candidates = find_candidates(
    pilot2_entities,
    terms
)

relationship_candidates = find_candidates(
    pilot2_relationships,
    terms
)

print("\nENTITY CANDIDATES:")
display(
    entity_candidates[
        ["title", "type", "frequency", "degree", "description"]
    ]
)

print("\nRELATIONSHIP CANDIDATES:")
display(
    relationship_candidates[
        ["source", "target", "description", "weight"]
    ]
)

M06: An average expansion rate of approximately 2 GW per year is needed to reach the 2040 PV potential.

ENTITY CANDIDATES:


,title,type,frequency,degree,description
6,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,POLICY,10,45,"The ""ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE"" (..."
152,AUSBBAUPOTENTIAL PHOTOVOLTAIK — 41 TWH BY 2040,TARGET,1,6,Target: PV expansion potential; Target value: ...
18,BÜRGER:INNEN,STAKEHOLDER,5,5,BÜRGER:INNEN refers to Austrian citizens ident...
42,PV-ANLAGEN,TECHNOLOGY,3,5,PV-ANLAGEN (photovoltaic installations or PV s...
66,GEBÄUDE,INFRASTRUCTURE,3,4,"GEBÄUDE (buildings), including existing and ne..."
7,KLIMANEUTRALITÄT IN ÖSTERREICH BIS 2040,TARGET,2,4,KLIMANEUTRALITÄT IN ÖSTERREICH BIS 2040 is an ...
329,KONSTANTER JÄHRLICHER PV-ZUBAU — CA. 2 GW P.A....,TARGET,1,4,Target: constant annual PV expansion; Target v...
427,KLIMANEUTRALITÄT 2040 (ÖSTERREICH),TARGET,1,4,Target: climate neutrality; Target value: 0; U...
160,INTEGRIERTER ÖSTERREICHISCHER NETZINFRASTRUKTU...,POLICY,1,3,Policy type: plan; Jurisdiction: Austria; Stat...
264,PV-AUSBAUZIELE 2040,TARGET,1,3,Target: PV expansion for 2040; Target value: n...



RELATIONSHIP CANDIDATES:


,source,target,description,weight
3,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,KLIMANEUTRALITÄT IN ÖSTERREICH BIS 2040,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLI...,9.0
6,ÖSTERREICH,KLIMANEUTRALITÄT IN ÖSTERREICH BIS 2040,Austria (ÖSTERREICH) has the official target o...,20.0
9,INTEGRIERTER ÖSTERREICHISCHER NETZINFRASTRUKTU...,PHOTOVOLTAIK-AUSBAUPOTENZIAL — 41 TWH BIS 2040,[RELATION_TYPE=SETS_TARGET] [MODALITY=EXPLICIT...,10.0
10,ÖSTERREICH,PHOTOVOLTAIK-AUSBAUPOTENZIAL — 41 TWH BIS 2040,[RELATION_TYPE=HAS_TARGET] [MODALITY=EXPLICIT_...,10.0
11,ÖSTERREICH,"JÄHRLICHER PV-ZUBAU — 1 GW (ÖSTERREICH, 2022)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
12,PHOTOVOLTAIK,"JÄHRLICHER PV-ZUBAU — 1 GW (ÖSTERREICH, 2022)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
13,ÖSTERREICH,"JÄHRLICHER PV-ZUBAU — 2,5 GW (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
14,PHOTOVOLTAIK,"JÄHRLICHER PV-ZUBAU — 2,5 GW (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
27,KLIMANEUTRALITÄT IN ÖSTERREICH BIS 2040,ENERGIEWENDE,[RELATION_TYPE=ASSOCIATED_WITH] [MODALITY=EXPL...,10.0
41,PHOTOVOLTAIK,KLIMANEUTRALITÄT IN ÖSTERREICH BIS 2040,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLI...,8.0


In [19]:
recall_checklist.loc[
    recall_checklist["fact_id"] == "M06",
    [
        "pilot2_entity_capture",
        "pilot2_relationship_capture",
        "pilot2_overall_classification",
        "pilot2_audit_notes",
    ],
] = [
    "FULL",
    "MISSING",
    "ENTITY_ONLY",
    (
        "The annual expansion target of approximately 2 GW per year and "
        "the 2040 PV potential are represented as entities, but no relationship "
        "directly connects the annual rate to achieving the 2040 potential."
    ),
]

In [20]:
fact_id = "M07"
terms = ["21 TWH", "2030", "TRANSITION 2040"]

fact = recall_checklist.loc[
    recall_checklist["fact_id"] == fact_id,
    "source_fact",
].iloc[0]

print(f"{fact_id}: {fact}")

entity_candidates = find_candidates(pilot2_entities, terms)
relationship_candidates = find_candidates(pilot2_relationships, terms)

print("\nENTITY CANDIDATES:")
display(
    entity_candidates[
        ["title", "type", "frequency", "degree", "description"]
    ]
)

print("\nRELATIONSHIP CANDIDATES:")
display(
    relationship_candidates[
        ["source", "target", "description", "weight"]
    ]
)

M07: The Transition 2040 scenario requires 21 TWh of photovoltaic electricity generation in 2030.

ENTITY CANDIDATES:


,title,type,frequency,degree,description
10,ERNEUERBAREN-AUSBAU-GESETZ (EAG),POLICY,5,10,"The Erneuerbaren-Ausbau-Gesetz (EAG), also kno..."
8,100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,TARGET,1,5,Target: nationally balanced renewable electric...
44,PV-AUSBAU VON MINDESTENS 11 TWH BIS 2030,TARGET,1,4,Target: PV electricity generation increase; Ta...
157,PV-STROMERZEUGUNG ERFORDERLICH — 21 TWH (ÖSTER...,TARGET,1,4,Target: required photovoltaic electricity gene...
227,100% NATIONAL BILANZIELL ERNEUERBARER GESAMTST...,TARGET,1,4,Target: complete national coverage of total el...
263,PV-AUSBAUZIELE 2030,TARGET,1,4,Target: PV expansion for 2030; Target value: n...
160,INTEGRIERTER ÖSTERREICHISCHER NETZINFRASTRUKTU...,POLICY,1,3,Policy type: plan; Jurisdiction: Austria; Stat...
264,PV-AUSBAUZIELE 2040,TARGET,1,3,Target: PV expansion for 2040; Target value: n...
158,PV-ZUWACHS GEGENÜBER 2020 ERFORDERLICH — 19 TW...,TARGET,1,2,Target: required PV expansion increase compare...
159,EAG GENANNTE ZUBAUMENGE PV BIS 2030 GEGENÜBER ...,TARGET,1,2,Target: PV expansion quantity stated in EAG co...



RELATIONSHIP CANDIDATES:


,source,target,description,weight
4,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLI...,9.0
5,PHOTOVOLTAIK,100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLI...,10.0
7,ÖSTERREICH,100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,[RELATION_TYPE=HAS_TARGET] [MODALITY=EXPLICIT_...,10.0
8,ERNEUERBAREN-AUSBAU-GESETZ (EAG),100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,[RELATION_TYPE=SETS_TARGET] [MODALITY=EXPLICIT...,10.0
15,ERNEUERBARE ENERGIEN,100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLI...,10.0
36,PHOTOVOLTAIK-STRATEGIE,PV-AUSBAU VON MINDESTENS 11 TWH BIS 2030,[RELATION_TYPE=SETS_TARGET] [MODALITY=EXPLICIT...,10.0
37,ERNEUERBAREN-AUSBAU-GESETZ (EAG),PV-AUSBAU VON MINDESTENS 11 TWH BIS 2030,[RELATION_TYPE=SETS_TARGET] [MODALITY=EXPLICIT...,10.0
38,ERNEUERBAREN-AUSBAU-GESETZ (EAG),100% NATIONAL BILANZIERTER ERNEUERBARER STROMV...,[RELATION_TYPE=SETS_TARGET] [MODALITY=EXPLICIT...,10.0
39,ÖSTERREICH,PV-AUSBAU VON MINDESTENS 11 TWH BIS 2030,[RELATION_TYPE=HAS_TARGET] [MODALITY=EXPLICIT_...,10.0
40,PHOTOVOLTAIK,PV-AUSBAU VON MINDESTENS 11 TWH BIS 2030,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLI...,9.0


In [21]:
mask = recall_checklist["fact_id"] == "M07"

recall_checklist.loc[mask, "pilot2_entity_capture"] = "FULL"
recall_checklist.loc[mask, "pilot2_relationship_capture"] = "PARTIAL"
recall_checklist.loc[mask, "pilot2_overall_classification"] = "PARTIALLY_STRUCTURED"
recall_checklist.loc[mask, "pilot2_audit_notes"] = (
    "The Transition 2040 scenario and the 21 TWh PV-generation target for "
    "2030 were extracted. However, the target is connected to Austria rather "
    "than directly to the Transition 2040 scenario, so the relationship "
    "provenance is only partially represented."
)

recall_checklist.loc[
    mask,
    [
        "fact_id",
        "source_fact",
        "pilot2_entity_capture",
        "pilot2_relationship_capture",
        "pilot2_overall_classification",
        "pilot2_audit_notes",
    ],
]

,fact_id,source_fact,pilot2_entity_capture,pilot2_relationship_capture,pilot2_overall_classification,pilot2_audit_notes
16,M07,The Transition 2040 scenario requires 21 TWh o...,FULL,PARTIAL,PARTIALLY_STRUCTURED,The Transition 2040 scenario and the 21 TWh PV...


In [22]:
fact_id = "M08"
terms = ["19 TWH", "2020", "2030"]

fact = recall_checklist.loc[
    recall_checklist["fact_id"] == fact_id,
    "source_fact",
].iloc[0]

print(f"{fact_id}: {fact}")

entity_candidates = find_candidates(pilot2_entities, terms)
relationship_candidates = find_candidates(pilot2_relationships, terms)

print("\nENTITY CANDIDATES:")
display(
    entity_candidates[
        ["title", "type", "frequency", "degree", "description"]
    ]
)

print("\nRELATIONSHIP CANDIDATES:")
display(
    relationship_candidates[
        ["source", "target", "description", "weight"]
    ]
)

M08: The 21 TWh requirement for 2030 represents an increase of 19 TWh compared with 2020.

ENTITY CANDIDATES:


,title,type,frequency,degree,description
10,ERNEUERBAREN-AUSBAU-GESETZ (EAG),POLICY,5,10,"The Erneuerbaren-Ausbau-Gesetz (EAG), also kno..."
8,100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,TARGET,1,5,Target: nationally balanced renewable electric...
44,PV-AUSBAU VON MINDESTENS 11 TWH BIS 2030,TARGET,1,4,Target: PV electricity generation increase; Ta...
157,PV-STROMERZEUGUNG ERFORDERLICH — 21 TWH (ÖSTER...,TARGET,1,4,Target: required photovoltaic electricity gene...
227,100% NATIONAL BILANZIELL ERNEUERBARER GESAMTST...,TARGET,1,4,Target: complete national coverage of total el...
263,PV-AUSBAUZIELE 2030,TARGET,1,4,Target: PV expansion for 2030; Target value: n...
264,PV-AUSBAUZIELE 2040,TARGET,1,3,Target: PV expansion for 2040; Target value: n...
158,PV-ZUWACHS GEGENÜBER 2020 ERFORDERLICH — 19 TW...,TARGET,1,2,Target: required PV expansion increase compare...
159,EAG GENANNTE ZUBAUMENGE PV BIS 2030 GEGENÜBER ...,TARGET,1,2,Target: PV expansion quantity stated in EAG co...
5,ERNEUERBARE ENERGIEN,TECHNOLOGY,1,1,Renewable energies (plural) are the set of ene...



RELATIONSHIP CANDIDATES:


,source,target,description,weight
4,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLI...,9.0
5,PHOTOVOLTAIK,100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLI...,10.0
7,ÖSTERREICH,100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,[RELATION_TYPE=HAS_TARGET] [MODALITY=EXPLICIT_...,10.0
8,ERNEUERBAREN-AUSBAU-GESETZ (EAG),100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,[RELATION_TYPE=SETS_TARGET] [MODALITY=EXPLICIT...,10.0
15,ERNEUERBARE ENERGIEN,100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLI...,10.0
36,PHOTOVOLTAIK-STRATEGIE,PV-AUSBAU VON MINDESTENS 11 TWH BIS 2030,[RELATION_TYPE=SETS_TARGET] [MODALITY=EXPLICIT...,10.0
37,ERNEUERBAREN-AUSBAU-GESETZ (EAG),PV-AUSBAU VON MINDESTENS 11 TWH BIS 2030,[RELATION_TYPE=SETS_TARGET] [MODALITY=EXPLICIT...,10.0
38,ERNEUERBAREN-AUSBAU-GESETZ (EAG),100% NATIONAL BILANZIERTER ERNEUERBARER STROMV...,[RELATION_TYPE=SETS_TARGET] [MODALITY=EXPLICIT...,10.0
39,ÖSTERREICH,PV-AUSBAU VON MINDESTENS 11 TWH BIS 2030,[RELATION_TYPE=HAS_TARGET] [MODALITY=EXPLICIT_...,10.0
40,PHOTOVOLTAIK,PV-AUSBAU VON MINDESTENS 11 TWH BIS 2030,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLI...,9.0


In [23]:
mask = recall_checklist["fact_id"] == "M08"

recall_checklist.loc[mask, "pilot2_entity_capture"] = "FULL"
recall_checklist.loc[mask, "pilot2_relationship_capture"] = "FULL"
recall_checklist.loc[mask, "pilot2_overall_classification"] = "FULLY_STRUCTURED"

recall_checklist.loc[mask, "pilot2_audit_notes"] = (
    "Pilot 2 extracted the 21 TWh requirement for 2030 and the "
    "required 19 TWh increase compared with 2020. A direct explicit-fact "
    "relationship connects the 21 TWh requirement with the 19 TWh increase."
)

recall_checklist.loc[
    mask,
    [
        "fact_id",
        "source_fact",
        "pilot2_entity_capture",
        "pilot2_relationship_capture",
        "pilot2_overall_classification",
        "pilot2_audit_notes",
    ],
]

,fact_id,source_fact,pilot2_entity_capture,pilot2_relationship_capture,pilot2_overall_classification,pilot2_audit_notes
17,M08,The 21 TWh requirement for 2030 represents an ...,FULL,FULL,FULLY_STRUCTURED,Pilot 2 extracted the 21 TWh requirement for 2...


In [24]:
fact_id = "M09"
terms = ["600 MIO", "600 MILLION", "2023"]

fact = recall_checklist.loc[
    recall_checklist["fact_id"] == fact_id,
    "source_fact",
].iloc[0]

print(f"{fact_id}: {fact}")

entity_candidates = find_candidates(pilot2_entities, terms)
relationship_candidates = find_candidates(pilot2_relationships, terms)

print("\nENTITY CANDIDATES:")
display(
    entity_candidates[
        ["title", "type", "frequency", "degree", "description"]
    ]
)

print("\nRELATIONSHIP CANDIDATES:")
display(
    relationship_candidates[
        ["source", "target", "description", "weight"]
    ]
)

M09: Approximately EUR 600 million was provided for PV expansion in 2023 through the Renewable Expansion Act and Climate and Energy Fund.

ENTITY CANDIDATES:


,title,type,frequency,degree,description
32,KLIMA- UND ENERGIEFONDS,ORGANIZATION,5,8,The KLIMA- UND ENERGIEFONDS (Climate and Energ...
34,E-CONTROL,ORGANIZATION,5,8,"E-CONTROL, also referred to as E-Control, is t..."
47,"PV-GENERATION — 6.3 TWH (ÖSTERREICH, 2023)",MARKET_METRIC,1,3,Metric: PV electricity generation; Value: 6.3;...
48,"PV-FÖRDERUNG — 600 MIO. EUR (ÖSTERREICH, 2023)",MARKET_METRIC,1,3,Metric: PV support/funding; Value: 600; Unit: ...
13,"JÄHRLICHER PV-ZUBAU — 2,5 GW (ÖSTERREICH, 2023)",MARKET_METRIC,1,2,Metric: annual PV capacity expansion; Value: 2...
79,ANTEIL FOSSILE ENERGIETRÄGER AM ENERGIEEINSATZ...,MARKET_METRIC,1,2,Metric: share of fossil energy carriers in tot...
81,"PV-GENERIERTE ENERGIE — 6,3 TWH (ÖSTERREICH, 2...",MARKET_METRIC,1,2,Metric: PV-generated energy; Value: 6.3; Unit:...
154,PV-ANTEIL AM STROMENDVERBRAUCH — CA. 10% (ÖSTE...,MARKET_METRIC,1,2,Metric: share of photovoltaics in final electr...
155,PV-ANTEIL AM ENERGIEBEDARF — CA. 2% (ÖSTERREIC...,MARKET_METRIC,1,2,Metric: share of photovoltaics in energy deman...
163,REKORDMARKEN PV-NEUINSTALLATION — 1 GW (2022);...,MARKET_METRIC,1,2,Metric: PV new installations; Value: 1 GW in 2...



RELATIONSHIP CANDIDATES:


,source,target,description,weight
13,ÖSTERREICH,"JÄHRLICHER PV-ZUBAU — 2,5 GW (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
14,PHOTOVOLTAIK,"JÄHRLICHER PV-ZUBAU — 2,5 GW (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
50,ÖSTERREICH,"PV-GENERATION — 6.3 TWH (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
51,PV-ANLAGEN,"PV-GENERATION — 6.3 TWH (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
52,PHOTOVOLTAIK,"PV-GENERATION — 6.3 TWH (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
53,ÖSTERREICH,"PV-FÖRDERUNG — 600 MIO. EUR (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
54,KLIMA- UND ENERGIEFONDS,"PV-FÖRDERUNG — 600 MIO. EUR (ÖSTERREICH, 2023)",[RELATION_TYPE=FUNDS] [MODALITY=EXPLICIT_FACT]...,9.0
55,ERNEUERBAREN-AUSBAU-GESETZ (EAG),"PV-FÖRDERUNG — 600 MIO. EUR (ÖSTERREICH, 2023)",[RELATION_TYPE=FUNDS] [MODALITY=EXPLICIT_FACT]...,9.0
79,FOSSILE ENERGIETRÄGER,ANTEIL FOSSILE ENERGIETRÄGER AM ENERGIEEINSATZ...,ANTEIL FOSSILE ENERGIETRÄGER AM ENERGIEEINSATZ...,20.0
80,ÖSTERREICH,ANTEIL FOSSILE ENERGIETRÄGER AM ENERGIEEINSATZ...,[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0


In [25]:
mask = recall_checklist["fact_id"] == "M09"

recall_checklist.loc[mask, "pilot2_entity_capture"] = "FULL"
recall_checklist.loc[mask, "pilot2_relationship_capture"] = "FULL"
recall_checklist.loc[mask, "pilot2_overall_classification"] = "FULLY_STRUCTURED"

recall_checklist.loc[mask, "pilot2_audit_notes"] = (
    "Pilot 2 extracted the EUR 600 million PV funding metric for Austria "
    "in 2023. Explicit FUNDS relationships connect both the Climate and "
    "Energy Fund and the Renewable Expansion Act to this funding metric."
)

recall_checklist.loc[
    mask,
    [
        "fact_id",
        "source_fact",
        "pilot2_entity_capture",
        "pilot2_relationship_capture",
        "pilot2_overall_classification",
        "pilot2_audit_notes",
    ],
]

,fact_id,source_fact,pilot2_entity_capture,pilot2_relationship_capture,pilot2_overall_classification,pilot2_audit_notes
18,M09,Approximately EUR 600 million was provided for...,FULL,FULL,FULLY_STRUCTURED,Pilot 2 extracted the EUR 600 million PV fundi...


In [26]:
mask = recall_checklist["fact_id"] == "M09"

recall_checklist.loc[mask, "pilot2_entity_capture"] = "FULL"
recall_checklist.loc[mask, "pilot2_relationship_capture"] = "FULL"
recall_checklist.loc[mask, "pilot2_overall_classification"] = "FULLY_STRUCTURED"

recall_checklist.loc[mask, "pilot2_audit_notes"] = (
    "Pilot 2 extracted the EUR 600 million PV funding metric for Austria "
    "in 2023. Explicit FUNDS relationships connect both the Climate and "
    "Energy Fund and the Renewable Expansion Act to this funding metric."
)

recall_checklist.loc[
    mask,
    [
        "fact_id",
        "source_fact",
        "pilot2_entity_capture",
        "pilot2_relationship_capture",
        "pilot2_overall_classification",
        "pilot2_audit_notes",
    ],
]

,fact_id,source_fact,pilot2_entity_capture,pilot2_relationship_capture,pilot2_overall_classification,pilot2_audit_notes
18,M09,Approximately EUR 600 million was provided for...,FULL,FULL,FULLY_STRUCTURED,Pilot 2 extracted the EUR 600 million PV fundi...


In [27]:
fact_id = "M10"
terms = ["10%", "2%", "2023"]

fact = recall_checklist.loc[
    recall_checklist["fact_id"] == fact_id,
    "source_fact",
].iloc[0]

print(f"{fact_id}: {fact}")
print("Search terms:", terms)

entity_candidates = find_candidates(
    pilot2_entities,
    terms
)

relationship_candidates = find_candidates(
    pilot2_relationships,
    terms
)

print("\nENTITY CANDIDATES:")
display(
    entity_candidates[
        ["title", "type", "frequency", "degree", "description"]
    ]
)

print("\nRELATIONSHIP CANDIDATES:")
display(
    relationship_candidates[
        ["source", "target", "description", "weight"]
    ]
)

M10: At the end of 2023, PV supplied almost 10% of Austria's final electricity consumption and approximately 2% of total energy demand.
Search terms: ['10%', '2%', '2023']

ENTITY CANDIDATES:


,title,type,frequency,degree,description
32,KLIMA- UND ENERGIEFONDS,ORGANIZATION,5,8,The KLIMA- UND ENERGIEFONDS (Climate and Energ...
34,E-CONTROL,ORGANIZATION,5,8,"E-CONTROL, also referred to as E-Control, is t..."
47,"PV-GENERATION — 6.3 TWH (ÖSTERREICH, 2023)",MARKET_METRIC,1,3,Metric: PV electricity generation; Value: 6.3;...
48,"PV-FÖRDERUNG — 600 MIO. EUR (ÖSTERREICH, 2023)",MARKET_METRIC,1,3,Metric: PV support/funding; Value: 600; Unit: ...
13,"JÄHRLICHER PV-ZUBAU — 2,5 GW (ÖSTERREICH, 2023)",MARKET_METRIC,1,2,Metric: annual PV capacity expansion; Value: 2...
79,ANTEIL FOSSILE ENERGIETRÄGER AM ENERGIEEINSATZ...,MARKET_METRIC,1,2,Metric: share of fossil energy carriers in tot...
81,"PV-GENERIERTE ENERGIE — 6,3 TWH (ÖSTERREICH, 2...",MARKET_METRIC,1,2,Metric: PV-generated energy; Value: 6.3; Unit:...
154,PV-ANTEIL AM STROMENDVERBRAUCH — CA. 10% (ÖSTE...,MARKET_METRIC,1,2,Metric: share of photovoltaics in final electr...
155,PV-ANTEIL AM ENERGIEBEDARF — CA. 2% (ÖSTERREIC...,MARKET_METRIC,1,2,Metric: share of photovoltaics in energy deman...
163,REKORDMARKEN PV-NEUINSTALLATION — 1 GW (2022);...,MARKET_METRIC,1,2,Metric: PV new installations; Value: 1 GW in 2...



RELATIONSHIP CANDIDATES:


,source,target,description,weight
13,ÖSTERREICH,"JÄHRLICHER PV-ZUBAU — 2,5 GW (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
14,PHOTOVOLTAIK,"JÄHRLICHER PV-ZUBAU — 2,5 GW (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
50,ÖSTERREICH,"PV-GENERATION — 6.3 TWH (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
51,PV-ANLAGEN,"PV-GENERATION — 6.3 TWH (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
52,PHOTOVOLTAIK,"PV-GENERATION — 6.3 TWH (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
53,ÖSTERREICH,"PV-FÖRDERUNG — 600 MIO. EUR (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
54,KLIMA- UND ENERGIEFONDS,"PV-FÖRDERUNG — 600 MIO. EUR (ÖSTERREICH, 2023)",[RELATION_TYPE=FUNDS] [MODALITY=EXPLICIT_FACT]...,9.0
55,ERNEUERBAREN-AUSBAU-GESETZ (EAG),"PV-FÖRDERUNG — 600 MIO. EUR (ÖSTERREICH, 2023)",[RELATION_TYPE=FUNDS] [MODALITY=EXPLICIT_FACT]...,9.0
79,FOSSILE ENERGIETRÄGER,ANTEIL FOSSILE ENERGIETRÄGER AM ENERGIEEINSATZ...,ANTEIL FOSSILE ENERGIETRÄGER AM ENERGIEEINSATZ...,20.0
80,ÖSTERREICH,ANTEIL FOSSILE ENERGIETRÄGER AM ENERGIEEINSATZ...,[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0


In [28]:
mask = recall_checklist["fact_id"] == "M10"

recall_checklist.loc[mask, "pilot2_entity_capture"] = "FULL"
recall_checklist.loc[mask, "pilot2_relationship_capture"] = "FULL"
recall_checklist.loc[mask, "pilot2_overall_classification"] = "FULLY_STRUCTURED"

recall_checklist.loc[mask, "pilot2_audit_notes"] = (
    "Pilot 2 extracted separate MARKET_METRIC entities for the approximately "
    "10% share of final electricity consumption and approximately 2% share "
    "of total energy demand in Austria in 2023. Explicit MEASURED_BY "
    "relationships connect Austria and photovoltaics to these metrics."
)

recall_checklist.loc[
    mask,
    [
        "fact_id",
        "source_fact",
        "pilot2_entity_capture",
        "pilot2_relationship_capture",
        "pilot2_overall_classification",
        "pilot2_audit_notes",
    ],
]

,fact_id,source_fact,pilot2_entity_capture,pilot2_relationship_capture,pilot2_overall_classification,pilot2_audit_notes
19,M10,"At the end of 2023, PV supplied almost 10% of ...",FULL,FULL,FULLY_STRUCTURED,Pilot 2 extracted separate MARKET_METRIC entit...


In [29]:
fact_id = "T01"

terms = [
    "DIGITALISIERUNG",
    "VERTEILNETZ",
    "STROMNETZ",
    "DEZENTRAL",
]

fact = recall_checklist.loc[
    recall_checklist["fact_id"] == fact_id,
    "source_fact",
].iloc[0]

print(f"{fact_id}: {fact}")
print("Search terms:", terms)

entity_candidates = find_candidates(
    pilot2_entities,
    terms,
)

relationship_candidates = find_candidates(
    pilot2_relationships,
    terms,
)

print("\nENTITY CANDIDATES:")
display(
    entity_candidates[
        ["title", "type", "frequency", "degree", "description"]
    ].head(20)
)

print("\nRELATIONSHIP CANDIDATES:")
display(
    relationship_candidates[
        ["source", "target", "description", "weight"]
    ].head(25)
)

T01: Digitalization of the distribution grid is important for integrating renewable and decentralized electricity.
Search terms: ['DIGITALISIERUNG', 'VERTEILNETZ', 'STROMNETZ', 'DEZENTRAL']

ENTITY CANDIDATES:


,title,type,frequency,degree,description
101,STROMNETZ,INFRASTRUCTURE,3,8,The electricity grid (Stromnetz) is the infras...
177,STROMNETZE,INFRASTRUCTURE,2,5,STROMNETZE refers to electricity grids in Aust...
294,DEZENTRALE ERZEUGUNGSSTRUKTUR,CONSTRAINT,1,4,Constraint type: structural/technical; Geograp...
468,DEZENTRALE NUTZUNG ERNEUERBARER ENERGIE,TECHNOLOGY,1,4,"Decentralized use of renewable energy, a goal ..."
483,DIGITALISIERUNG DER MARKTKOMMUNIKATION,INFRASTRUCTURE,1,3,Digitalization of market communication refers ...
54,WEITERENTWICKLUNG DER ÖFFENTLICHEN STROMNETZE,POLICY,1,2,Policy type: strategy/action field; Jurisdicti...
278,VERTEILNETZ,INFRASTRUCTURE,1,2,The electricity distribution grid (Verteilnetz...
40,ÖFFENTLICHE STROMNETZE,INFRASTRUCTURE,1,1,Public electricity grids in Austria are cited ...
105,DIGITALISIERUNG,TECHNOLOGY,1,1,Digitalization refers to enabling technologies...
130,VOLKSWIRTSCHAFTLICHES OPTIMUM DER STROMNETZE,CONSTRAINT,1,1,The economic optimum of power grids is a const...



RELATIONSHIP CANDIDATES:


,source,target,description,weight
44,PHOTOVOLTAIK-STRATEGIE,WEITERENTWICKLUNG DER ÖFFENTLICHEN STROMNETZE,[RELATION_TYPE=PROPOSES] [MODALITY=EXPLICIT_FA...,8.0
59,ÖFFENTLICHE STROMNETZE,WEITERENTWICKLUNG DER ÖFFENTLICHEN STROMNETZE,[RELATION_TYPE=ASSOCIATED_WITH] [MODALITY=EXPL...,9.0
129,PV-ANLAGE,STROMNETZ,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,9.0
130,DIGITALISIERUNG,STROMNETZ,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,9.0
148,VOLKSWIRTSCHAFTLICHES OPTIMUM DER STROMNETZE,STROMNETZ,[RELATION_TYPE=CONSTRAINS] [MODALITY=EXPLICIT_...,8.0
150,PV-ANLAGEN IN UNTERSCHIEDLICHER AUSRICHTUNG UN...,STROMNETZ,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,8.0
189,FRAGE DER ANBINDUNG AN DAS ÖFFENTLICHE STROMNETZ,PHOTOVOLTAIK,[RELATION_TYPE=CONSTRAINS] [MODALITY=EXPLICIT_...,10.0
204,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,STROMNETZE,[RELATION_TYPE=PROPOSES] [MODALITY=EXPLICIT_FA...,10.0
222,AKZEPTANZ,STROMNETZE,[RELATION_TYPE=CONSTRAINS] [MODALITY=EXPLICIT_...,10.0
312,STROMNETZ,PV-ANLAGE,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,9.0


In [30]:
mask = recall_checklist["fact_id"] == "T01"

recall_checklist.loc[mask, "pilot2_entity_capture"] = "FULL"
recall_checklist.loc[mask, "pilot2_relationship_capture"] = "PARTIAL"
recall_checklist.loc[mask, "pilot2_overall_classification"] = "PARTIALLY_STRUCTURED"

recall_checklist.loc[mask, "pilot2_audit_notes"] = (
    "Pilot 2 extracted DIGITALISIERUNG, STROMNETZ, VERTEILNETZ and "
    "decentralized-energy concepts. It also created a SUPPORTS relationship "
    "from DIGITALISIERUNG to STROMNETZ. However, the complete source claim "
    "linking digitalization of the distribution grid specifically to the "
    "integration of renewable and decentralized electricity is distributed "
    "across multiple graph records rather than represented as one complete relationship."
)

recall_checklist.loc[
    mask,
    [
        "fact_id",
        "source_fact",
        "pilot2_entity_capture",
        "pilot2_relationship_capture",
        "pilot2_overall_classification",
        "pilot2_audit_notes",
    ],
]

,fact_id,source_fact,pilot2_entity_capture,pilot2_relationship_capture,pilot2_overall_classification,pilot2_audit_notes
20,T01,Digitalization of the distribution grid is imp...,FULL,PARTIAL,PARTIALLY_STRUCTURED,"Pilot 2 extracted DIGITALISIERUNG, STROMNETZ, ..."


In [31]:
fact_id = "T02"

terms = [
    "PHOTOVOLTAIK-ERZEUGUNGSSPITZEN",
    "ERZEUGUNGSSPITZEN",
    "STROMNETZ",
    "ÖFFENTLICHE STROMNETZE",
]

fact = recall_checklist.loc[
    recall_checklist["fact_id"] == fact_id,
    "source_fact"
].iloc[0]

print("\n" + "=" * 90)
print(f"{fact_id}: {fact}")
print("Search terms:", terms)

entity_candidates = find_candidates(
    pilot2_entities,
    terms
)

relationship_candidates = find_candidates(
    pilot2_relationships,
    terms
)

print("\nENTITY CANDIDATES:")
display(
    entity_candidates[
        ["title", "type", "frequency", "degree", "description"]
    ].head(20)
)

print("\nRELATIONSHIP CANDIDATES:")
display(
    relationship_candidates[
        ["source", "target", "description", "weight"]
    ].head(25)
)


T02: Photovoltaic generation peaks can no longer always be completely absorbed by the public electricity grid.
Search terms: ['PHOTOVOLTAIK-ERZEUGUNGSSPITZEN', 'ERZEUGUNGSSPITZEN', 'STROMNETZ', 'ÖFFENTLICHE STROMNETZE']

ENTITY CANDIDATES:


,title,type,frequency,degree,description
101,STROMNETZ,INFRASTRUCTURE,3,8,The electricity grid (Stromnetz) is the infras...
293,PHOTOVOLTAIK-ERZEUGUNGSSPITZEN,CONSTRAINT,1,8,Constraint type: technical; Geographic scope: ...
177,STROMNETZE,INFRASTRUCTURE,2,5,STROMNETZE refers to electricity grids in Aust...
54,WEITERENTWICKLUNG DER ÖFFENTLICHEN STROMNETZE,POLICY,1,2,Policy type: strategy/action field; Jurisdicti...
40,ÖFFENTLICHE STROMNETZE,INFRASTRUCTURE,1,1,Public electricity grids in Austria are cited ...
130,VOLKSWIRTSCHAFTLICHES OPTIMUM DER STROMNETZE,CONSTRAINT,1,1,The economic optimum of power grids is a const...
165,FRAGE DER ANBINDUNG AN DAS ÖFFENTLICHE STROMNETZ,CONSTRAINT,1,1,Constraint: Question/challenge of grid connect...
283,DIGITALE INFRASTRUKTUR DES STROMNETZES,INFRASTRUCTURE,1,1,The digital infrastructure of the electricity ...



RELATIONSHIP CANDIDATES:


,source,target,description,weight
44,PHOTOVOLTAIK-STRATEGIE,WEITERENTWICKLUNG DER ÖFFENTLICHEN STROMNETZE,[RELATION_TYPE=PROPOSES] [MODALITY=EXPLICIT_FA...,8.0
59,ÖFFENTLICHE STROMNETZE,WEITERENTWICKLUNG DER ÖFFENTLICHEN STROMNETZE,[RELATION_TYPE=ASSOCIATED_WITH] [MODALITY=EXPL...,9.0
129,PV-ANLAGE,STROMNETZ,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,9.0
130,DIGITALISIERUNG,STROMNETZ,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,9.0
148,VOLKSWIRTSCHAFTLICHES OPTIMUM DER STROMNETZE,STROMNETZ,[RELATION_TYPE=CONSTRAINS] [MODALITY=EXPLICIT_...,8.0
150,PV-ANLAGEN IN UNTERSCHIEDLICHER AUSRICHTUNG UN...,STROMNETZ,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,8.0
189,FRAGE DER ANBINDUNG AN DAS ÖFFENTLICHE STROMNETZ,PHOTOVOLTAIK,[RELATION_TYPE=CONSTRAINS] [MODALITY=EXPLICIT_...,10.0
204,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,STROMNETZE,[RELATION_TYPE=PROPOSES] [MODALITY=EXPLICIT_FA...,10.0
222,AKZEPTANZ,STROMNETZE,[RELATION_TYPE=CONSTRAINS] [MODALITY=EXPLICIT_...,10.0
312,STROMNETZ,PV-ANLAGE,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,9.0


In [32]:
mask = recall_checklist["fact_id"] == "T02"

recall_checklist.loc[mask, "pilot2_entity_capture"] = "FULL"
recall_checklist.loc[mask, "pilot2_relationship_capture"] = "FULL"
recall_checklist.loc[mask, "pilot2_overall_classification"] = "FULLY_STRUCTURED"

recall_checklist.loc[mask, "pilot2_audit_notes"] = (
    "Pilot 2 extracted photovoltaic generation peaks as a CONSTRAINT "
    "and the electricity grid as INFRASTRUCTURE. An explicit CONSTRAINS "
    "relationship connects PHOTOVOLTAIK-ERZEUGUNGSSPITZEN to STROMNETZ."
)

recall_checklist.loc[
    mask,
    [
        "fact_id",
        "source_fact",
        "pilot2_entity_capture",
        "pilot2_relationship_capture",
        "pilot2_overall_classification",
        "pilot2_audit_notes",
    ],
]

,fact_id,source_fact,pilot2_entity_capture,pilot2_relationship_capture,pilot2_overall_classification,pilot2_audit_notes
21,T02,Photovoltaic generation peaks can no longer al...,FULL,FULL,FULLY_STRUCTURED,Pilot 2 extracted photovoltaic generation peak...


In [33]:
fact_id = "T03"

terms = [
    "ENERGIEMANAGEMENT",
    "INTELLIGENTES LASTMANAGEMENT",
    "SPEICHER",
    "ERZEUGUNGSSPITZEN",
    "VERBRAUCHSSPITZEN",
    "EIGENVERBRAUCH",
]

fact = recall_checklist.loc[
    recall_checklist["fact_id"] == fact_id,
    "source_fact"
].iloc[0]

print("\n" + "=" * 90)
print(f"{fact_id}: {fact}")
print("Search terms:", terms)

entity_candidates = find_candidates(pilot2_entities, terms)
relationship_candidates = find_candidates(pilot2_relationships, terms)

print("\nENTITY CANDIDATES:")
display(
    entity_candidates[
        ["title", "type", "frequency", "degree", "description"]
    ].head(25)
)

print("\nRELATIONSHIP CANDIDATES:")
display(
    relationship_candidates[
        ["source", "target", "description", "weight"]
    ].head(30)
)


T03: Smart local energy management and storage can manage generation and consumption peaks and increase self-consumption.
Search terms: ['ENERGIEMANAGEMENT', 'INTELLIGENTES LASTMANAGEMENT', 'SPEICHER', 'ERZEUGUNGSSPITZEN', 'VERBRAUCHSSPITZEN', 'EIGENVERBRAUCH']

ENTITY CANDIDATES:


,title,type,frequency,degree,description
293,PHOTOVOLTAIK-ERZEUGUNGSSPITZEN,CONSTRAINT,1,8,Constraint type: technical; Geographic scope: ...
122,PRIORITÄT FÜR DEN EIGENVERBRAUCH AUS PV-ANLAGE...,TARGET,1,4,Target: standard for maximum self-supply from ...
282,ENERGIEMANAGEMENT,INFRASTRUCTURE,1,4,"Energy management systems are referenced, espe..."
308,GROSSSPEICHER,INFRASTRUCTURE,1,4,Large-scale storage systems for long-term elec...
484,ENERGIEMANAGEMENTLÖSUNGEN FÜR HAUSHALT UND UNT...,TECHNOLOGY,1,4,"Smart, user-friendly energy-management solutio..."
97,MOBILE UND STATIONÄRE SPEICHER,TECHNOLOGY,1,2,Mobile and stationary storage units that are i...
98,ENERGIEMANAGEMENTSYSTEM,TECHNOLOGY,1,2,An energy management system is integrated in P...
141,VERHINDERUNG VON EINSPEISE- UND LASTSPITZEN DU...,TARGET,1,2,Target: prevention of feed-in and load peaks v...
148,SMARTE ENERGIEMANAGEMENTLÖSUNGEN,TECHNOLOGY,1,2,Smart energy management solutions that optimiz...
291,SPEICHERINITIATIVE DES KLIMA- UND ENERGIEFONDS,SUPPORT_SCHEME,1,2,Support scheme: funding initiative for storage...



RELATIONSHIP CANDIDATES:


,source,target,description,weight
119,MOBILE UND STATIONÄRE SPEICHER,PV-ANLAGE,[RELATION_TYPE=PART_OF] [MODALITY=EXPLICIT_FAC...,10.0
120,ENERGIEMANAGEMENTSYSTEM,PV-ANLAGE,[RELATION_TYPE=PART_OF] [MODALITY=EXPLICIT_FAC...,10.0
141,PRIORITÄT FÜR DEN EIGENVERBRAUCH AUS PV-ANLAGE...,PRIVATE UND UNTERNEHMERISCHE EINZELANLAGENBETR...,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,8.0
142,PRIORITÄT FÜR DEN EIGENVERBRAUCH AUS PV-ANLAGE...,ENERGIEGEMEINSCHAFTEN,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,8.0
143,PRIORITÄT FÜR DEN EIGENVERBRAUCH AUS PV-ANLAGE...,ALLE NEUEN UND GRUNDLEGEND SANIERTEN GEBÄUDE H...,[RELATION_TYPE=ASSOCIATED_WITH] [MODALITY=EXPL...,7.0
146,AKTIVER KUNDE,PRIORITÄT FÜR DEN EIGENVERBRAUCH AUS PV-ANLAGE...,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,9.0
156,ENERGIEMANAGEMENTSYSTEM,VERHINDERUNG VON EINSPEISE- UND LASTSPITZEN DU...,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,10.0
157,MOBILE UND STATIONÄRE SPEICHER,VERHINDERUNG VON EINSPEISE- UND LASTSPITZEN DU...,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,10.0
163,SMARTE ENERGIEMANAGEMENTLÖSUNGEN,HAUSHALT,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,10.0
164,SMARTE ENERGIEMANAGEMENTLÖSUNGEN,UNTERNEHMEN,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,10.0


In [34]:
mask = recall_checklist["fact_id"] == "T03"

recall_checklist.loc[mask, "pilot2_entity_capture"] = "FULL"
recall_checklist.loc[mask, "pilot2_relationship_capture"] = "PARTIAL"
recall_checklist.loc[mask, "pilot2_overall_classification"] = "PARTIALLY_STRUCTURED"

recall_checklist.loc[mask, "pilot2_audit_notes"] = (
    "Pilot 2 extracted smart energy management, storage technologies, "
    "peak prevention and self-consumption concepts. It also extracted "
    "relationships connecting energy-management systems and storage to "
    "the prevention of feed-in and load peaks. However, the complete source "
    "claim—including both peak management and increased self-consumption—"
    "is distributed across several records rather than represented by one "
    "complete relationship."
)

recall_checklist.loc[
    mask,
    [
        "fact_id",
        "source_fact",
        "pilot2_entity_capture",
        "pilot2_relationship_capture",
        "pilot2_overall_classification",
        "pilot2_audit_notes",
    ],
]

,fact_id,source_fact,pilot2_entity_capture,pilot2_relationship_capture,pilot2_overall_classification,pilot2_audit_notes
22,T03,Smart local energy management and storage can ...,FULL,PARTIAL,PARTIALLY_STRUCTURED,"Pilot 2 extracted smart energy management, sto..."


In [35]:
fact_id = "T04"

terms = [
    "PHOTOVOLTAIK",
    "E-MOBILITÄT",
    "WÄRMEPUMPEN",
    "WASSERSTOFF",
    "SEKTORENKOPPLUNG",
]

fact = recall_checklist.loc[
    recall_checklist["fact_id"] == fact_id,
    "source_fact"
].iloc[0]

print("\n" + "=" * 90)
print(f"{fact_id}: {fact}")
print("Search terms:", terms)

entity_candidates = find_candidates(pilot2_entities, terms)
relationship_candidates = find_candidates(pilot2_relationships, terms)

print("\nENTITY CANDIDATES:")
display(
    entity_candidates[
        ["title", "type", "frequency", "degree", "description"]
    ].head(25)
)

print("\nRELATIONSHIP CANDIDATES:")
display(
    relationship_candidates[
        ["source", "target", "description", "weight"]
    ].head(30)
)


T04: PV should operate synergistically with electric mobility, heat pumps, hydrogen and other generation and storage technologies.
Search terms: ['PHOTOVOLTAIK', 'E-MOBILITÄT', 'WÄRMEPUMPEN', 'WASSERSTOFF', 'SEKTORENKOPPLUNG']

ENTITY CANDIDATES:


,title,type,frequency,degree,description
3,PHOTOVOLTAIK,TECHNOLOGY,11,85,Photovoltaik (photovoltaics) is a renewable el...
6,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,POLICY,10,45,"The ""ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE"" (..."
31,PHOTOVOLTAIK-STRATEGIE,POLICY,1,21,Policy type: strategy; Jurisdiction: Austria; ...
347,PHOTOVOLTAIK (PV),TECHNOLOGY,2,18,Photovoltaics (PV) is a solar energy technolog...
180,PHOTOVOLTAIKANLAGEN,TECHNOLOGY,2,8,PHOTOVOLTAIKANLAGEN refers to photovoltaic ins...
293,PHOTOVOLTAIK-ERZEUGUNGSSPITZEN,CONSTRAINT,1,8,Constraint type: technical; Geographic scope: ...
152,AUSBBAUPOTENTIAL PHOTOVOLTAIK — 41 TWH BY 2040,TARGET,1,6,Target: PV expansion potential; Target value: ...
214,FÖRDERUNGEN FÜR PHOTOVOLTAIK,SUPPORT_SCHEME,1,3,Support schemes for photovoltaics include fina...
287,FÖRDERPROGRAMM MUSTER- & LEUCHTTURMPROJEKTE PH...,SUPPORT_SCHEME,1,3,Support scheme: Funding programme; Jurisdictio...
104,WASSERSTOFF,TECHNOLOGY,2,2,WASSERSTOFF (hydrogen) is referenced as a tech...



RELATIONSHIP CANDIDATES:


,source,target,description,weight
0,"BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENE...",ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,[RELATION_TYPE=RESPONSIBLE_FOR] [MODALITY=EXPL...,10.0
2,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,PHOTOVOLTAIK,The Austrian Photovoltaic Strategy (ÖSTERREICH...,30.0
3,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,KLIMANEUTRALITÄT IN ÖSTERREICH BIS 2040,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLI...,9.0
4,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLI...,9.0
5,PHOTOVOLTAIK,100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLI...,10.0
9,INTEGRIERTER ÖSTERREICHISCHER NETZINFRASTRUKTU...,PHOTOVOLTAIK-AUSBAUPOTENZIAL — 41 TWH BIS 2040,[RELATION_TYPE=SETS_TARGET] [MODALITY=EXPLICIT...,10.0
10,ÖSTERREICH,PHOTOVOLTAIK-AUSBAUPOTENZIAL — 41 TWH BIS 2040,[RELATION_TYPE=HAS_TARGET] [MODALITY=EXPLICIT_...,10.0
12,PHOTOVOLTAIK,"JÄHRLICHER PV-ZUBAU — 1 GW (ÖSTERREICH, 2022)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
14,PHOTOVOLTAIK,"JÄHRLICHER PV-ZUBAU — 2,5 GW (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
16,PHOTOVOLTAIK-ANLAGE,PHOTOVOLTAIK,[RELATION_TYPE=ASSOCIATED_WITH] [MODALITY=EXPL...,9.0


In [36]:
mask = recall_checklist["fact_id"] == "T04"

recall_checklist.loc[mask, "pilot2_entity_capture"] = "PARTIAL"
recall_checklist.loc[mask, "pilot2_relationship_capture"] = "MISSING"
recall_checklist.loc[mask, "pilot2_overall_classification"] = "ENTITY_ONLY"

recall_checklist.loc[mask, "pilot2_audit_notes"] = (
    "Pilot 2 extracted Photovoltaics, e-mobility, and hydrogen as TECHNOLOGY "
    "entities. However, heat pumps and sector coupling were not identified in "
    "the displayed candidates, and no relationship represents the stated "
    "synergistic operation among these technologies."
)

recall_checklist.loc[
    mask,
    [
        "fact_id",
        "source_fact",
        "pilot2_entity_capture",
        "pilot2_relationship_capture",
        "pilot2_overall_classification",
        "pilot2_audit_notes",
    ],
]

,fact_id,source_fact,pilot2_entity_capture,pilot2_relationship_capture,pilot2_overall_classification,pilot2_audit_notes
23,T04,PV should operate synergistically with electri...,PARTIAL,MISSING,ENTITY_ONLY,"Pilot 2 extracted Photovoltaics, e-mobility, a..."


In [37]:
fact_id = "T05"

terms = [
    "OST-WEST",
    "VERTIKAL",
    "WINTER",
    "ÜBERGANG",
    "PV-ANLAGEN",
]

fact = recall_checklist.loc[
    recall_checklist["fact_id"] == fact_id,
    "source_fact"
].iloc[0]

print("\n" + "=" * 90)
print(f"{fact_id}: {fact}")
print("Search terms:", terms)

entity_candidates = find_candidates(
    pilot2_entities,
    terms
)

relationship_candidates = find_candidates(
    pilot2_relationships,
    terms
)

print("\nENTITY CANDIDATES:")
display(
    entity_candidates[
        ["title", "type", "frequency", "degree", "description"]
    ].head(25)
)

print("\nRELATIONSHIP CANDIDATES:")
display(
    relationship_candidates[
        ["source", "target", "description", "weight"]
    ].head(30)
)


T05: East-west-oriented and vertical PV installations can provide higher PV yields during shoulder and winter periods.
Search terms: ['OST-WEST', 'VERTIKAL', 'WINTER', 'ÜBERGANG', 'PV-ANLAGEN']

ENTITY CANDIDATES:


,title,type,frequency,degree,description
180,PHOTOVOLTAIKANLAGEN,TECHNOLOGY,2,8,PHOTOVOLTAIKANLAGEN refers to photovoltaic ins...
42,PV-ANLAGEN,TECHNOLOGY,3,5,PV-ANLAGEN (photovoltaic installations or PV s...
122,PRIORITÄT FÜR DEN EIGENVERBRAUCH AUS PV-ANLAGE...,TARGET,1,4,Target: standard for maximum self-supply from ...
323,UMSATZSTEUERBEFREIUNG FÜR PV-ANLAGEN BIS 35 KW...,SUPPORT_SCHEME,1,3,Zero-percent VAT for PV systems up to 35 kWp i...
327,WIRTSCHAFTLICHKEIT DES PV-ANLAGENBETRIEBS,CONSTRAINT,1,3,"Economic viability of PV operations, stated as..."
332,STEUERLICHE ASPEKTE FÜR PV-ANLAGEN,POLICY,1,3,Policy type: fiscal/tax measure; Jurisdiction:...
354,PV-ANLAGEN AN LÄRMSCHUTZWÄNDEN,TECHNOLOGY,1,3,PV system deployment on noise barriers in road...
119,KOSTENGÜNSTIGER STROM AUS FREIFLÄCHEN-BIODIVER...,MARKET_METRIC,1,2,Metric: affordability of electricity generated...
245,AGRI-PV-ANLAGEN,TECHNOLOGY,1,2,Agri-PV systems are installations enabling com...
272,"PRODUKTQUALITÄT, INSTALLATION, SERVICE, REPARA...",SUPPORT_SCHEME,1,2,Support scheme involving measures to assure qu...



RELATIONSHIP CANDIDATES:


,source,target,description,weight
45,PHOTOVOLTAIK-STRATEGIE,WIRTSCHAFTLICHKEIT DER PV-ANLAGEN,[RELATION_TYPE=PROPOSES] [MODALITY=EXPLICIT_FA...,8.0
51,PV-ANLAGEN,"PV-GENERATION — 6.3 TWH (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
89,PV-ANLAGEN,GEBÄUDE,[RELATION_TYPE=LOCATED_IN] [MODALITY=EXPLICIT_...,9.0
90,PV-ANLAGEN,STROMERZEUGUNGSFUNKTION ALLER NEUEN UND GRUNDS...,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=SCENA...,9.0
113,BIODIVERSITÄTS-SOLARPARK,KOSTENGÜNSTIGER STROM AUS FREIFLÄCHEN-BIODIVER...,[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
114,AGRI-PV-ANLAGE,KOSTENGÜNSTIGER STROM AUS FREIFLÄCHEN-BIODIVER...,[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
141,PRIORITÄT FÜR DEN EIGENVERBRAUCH AUS PV-ANLAGE...,PRIVATE UND UNTERNEHMERISCHE EINZELANLAGENBETR...,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,8.0
142,PRIORITÄT FÜR DEN EIGENVERBRAUCH AUS PV-ANLAGE...,ENERGIEGEMEINSCHAFTEN,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,8.0
143,PRIORITÄT FÜR DEN EIGENVERBRAUCH AUS PV-ANLAGE...,ALLE NEUEN UND GRUNDLEGEND SANIERTEN GEBÄUDE H...,[RELATION_TYPE=ASSOCIATED_WITH] [MODALITY=EXPL...,7.0
146,AKTIVER KUNDE,PRIORITÄT FÜR DEN EIGENVERBRAUCH AUS PV-ANLAGE...,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,9.0


In [38]:
mask = recall_checklist["fact_id"] == "T05"

recall_checklist.loc[mask, "pilot2_entity_capture"] = "PARTIAL"
recall_checklist.loc[mask, "pilot2_relationship_capture"] = "MISSING"
recall_checklist.loc[mask, "pilot2_overall_classification"] = "ENTITY_ONLY"

recall_checklist.loc[mask, "pilot2_audit_notes"] = (
    "Pilot 2 extracted a general TECHNOLOGY entity for PV installations "
    "in different orientations. However, east-west orientation, vertical "
    "orientation, and their higher yields during shoulder and winter periods "
    "were not represented precisely. No relationship captures the seasonal "
    "yield benefit."
)

recall_checklist.loc[
    mask,
    [
        "fact_id",
        "source_fact",
        "pilot2_entity_capture",
        "pilot2_relationship_capture",
        "pilot2_overall_classification",
        "pilot2_audit_notes",
    ],
]

,fact_id,source_fact,pilot2_entity_capture,pilot2_relationship_capture,pilot2_overall_classification,pilot2_audit_notes
24,T05,East-west-oriented and vertical PV installatio...,PARTIAL,MISSING,ENTITY_ONLY,Pilot 2 extracted a general TECHNOLOGY entity ...


In [39]:
fact_id = "T06"

terms = [
    "AGRI-PV",
    "LANDWIRTSCHAFT",
    "DOPPELNUTZUNG",
    "FLÄCHENEFFIZIENZ",
    "FLÄCHENNUTZUNG",
]

fact = recall_checklist.loc[
    recall_checklist["fact_id"] == fact_id,
    "source_fact"
].iloc[0]

print("\n" + "=" * 90)
print(f"{fact_id}: {fact}")
print("Search terms:", terms)

entity_candidates = find_candidates(
    pilot2_entities,
    terms
)

relationship_candidates = find_candidates(
    pilot2_relationships,
    terms
)

print("\nENTITY CANDIDATES:")
display(
    entity_candidates[
        ["title", "type", "frequency", "degree", "description"]
    ].head(25)
)

print("\nRELATIONSHIP CANDIDATES:")
display(
    relationship_candidates[
        ["source", "target", "description", "weight"]
    ].head(30)
)


T06: Agri-PV enables combined agricultural and photovoltaic use and can increase land-use efficiency.
Search terms: ['AGRI-PV', 'LANDWIRTSCHAFT', 'DOPPELNUTZUNG', 'FLÄCHENEFFIZIENZ', 'FLÄCHENNUTZUNG']

ENTITY CANDIDATES:


,title,type,frequency,degree,description
10,ERNEUERBAREN-AUSBAU-GESETZ (EAG),POLICY,5,10,"The Erneuerbaren-Ausbau-Gesetz (EAG), also kno..."
93,AGRI-PV-ANLAGE,TECHNOLOGY,3,7,AGRI-PV-ANLAGE refers to a form of photovoltai...
236,LANDWIRTSCHAFT,STAKEHOLDER,2,2,"LANDWIRTSCHAFT, the agricultural sector, is ad..."
119,KOSTENGÜNSTIGER STROM AUS FREIFLÄCHEN-BIODIVER...,MARKET_METRIC,1,2,Metric: affordability of electricity generated...
151,INVESTITIONSZUSCHUSS FÜR AGRI-PV ANLAGEN,SUPPORT_SCHEME,1,2,Investment subsidy provided for Agri-PV system...
183,FÖRDERUNGEN FÜR DOPPELNUTZUNGEN UND BEVORZUGTE...,SUPPORT_SCHEME,1,2,Increased incentives for dual-use and other pr...
245,AGRI-PV-ANLAGEN,TECHNOLOGY,1,2,Agri-PV systems are installations enabling com...
365,"MEHRFACHNUTZUNG BEI BAULICHEN INFRASTRUKTUREN,...",CONSTRAINT,1,2,"Multiple uses of built infrastructure, agricul..."
114,LANDWIRT:INNEN,STAKEHOLDER,1,1,"Farmers benefit from Agri-PV installations, ma..."
181,DOPPELNUTZUNGEN UND ANDERE BEVORZUGTE PV-ANWEN...,TECHNOLOGY,1,1,"Dual-use and other preferred PV applications, ..."



RELATIONSHIP CANDIDATES:


,source,target,description,weight
113,BIODIVERSITÄTS-SOLARPARK,KOSTENGÜNSTIGER STROM AUS FREIFLÄCHEN-BIODIVER...,[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
114,AGRI-PV-ANLAGE,KOSTENGÜNSTIGER STROM AUS FREIFLÄCHEN-BIODIVER...,[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
136,AGRI-PV-ANLAGE,LANDWIRT:INNEN,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,9.0
165,ERNEUERBAREN-AUSBAU-GESETZ (EAG),INVESTITIONSZUSCHUSS FÜR AGRI-PV ANLAGEN,[RELATION_TYPE=PROPOSES] [MODALITY=EXPLICIT_FA...,10.0
166,INVESTITIONSZUSCHUSS FÜR AGRI-PV ANLAGEN,PHOTOVOLTAIK,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,9.0
213,ERNEUERBAREN-AUSBAU-GESETZ,FÖRDERUNGEN FÜR DOPPELNUTZUNGEN UND BEVORZUGTE...,[RELATION_TYPE=PROPOSES] [MODALITY=EXPLICIT_FA...,10.0
214,FÖRDERUNGEN FÜR DOPPELNUTZUNGEN UND BEVORZUGTE...,DOPPELNUTZUNGEN UND ANDERE BEVORZUGTE PV-ANWEN...,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,10.0
257,AGRI-PV-ANLAGE,FOTOVOLTAIKANLAGE,[RELATION_TYPE=PART_OF] [MODALITY=EXPLICIT_FAC...,10.0
258,AGRI-PV-ANLAGE,LANDWIRTSCHAFT,[RELATION_TYPE=ASSOCIATED_WITH] [MODALITY=EXPL...,8.0
280,DOPPELNUTZUNG MIT AGRI-PV,AGRI-PV-ANLAGE,[RELATION_TYPE=ALIAS_OF] [MODALITY=EXPLICIT_FA...,10.0


In [40]:
mask = recall_checklist["fact_id"] == "T06"

recall_checklist.loc[mask, "pilot2_entity_capture"] = "FULL"
recall_checklist.loc[mask, "pilot2_relationship_capture"] = "PARTIAL"
recall_checklist.loc[mask, "pilot2_overall_classification"] = "PARTIALLY_STRUCTURED"

recall_checklist.loc[mask, "pilot2_audit_notes"] = (
    "Pilot 2 extracted Agri-PV, agriculture, farmers, dual land use, "
    "and several relevant relationships. It clearly represents the "
    "combination of agricultural and photovoltaic use. However, the "
    "specific claim that Agri-PV increases land-use efficiency is not "
    "represented as a precise direct relationship."
)

recall_checklist.loc[
    mask,
    [
        "fact_id",
        "source_fact",
        "pilot2_entity_capture",
        "pilot2_relationship_capture",
        "pilot2_overall_classification",
        "pilot2_audit_notes",
    ],
]

,fact_id,source_fact,pilot2_entity_capture,pilot2_relationship_capture,pilot2_overall_classification,pilot2_audit_notes
25,T06,Agri-PV enables combined agricultural and phot...,FULL,PARTIAL,PARTIALLY_STRUCTURED,"Pilot 2 extracted Agri-PV, agriculture, farmer..."


In [41]:
fact_id = "T07"

terms = [
    "FREIFLÄCHEN",
    "BIODIVERSITÄT",
    "BIODIVERSITÄTS-PV",
    "ÖKOLOGISCH",
    "NATURSCHUTZ",
]

fact = recall_checklist.loc[
    recall_checklist["fact_id"] == fact_id,
    "source_fact"
].iloc[0]

print("\n" + "=" * 90)
print(f"{fact_id}: {fact}")
print("Search terms:", terms)

entity_candidates = find_candidates(
    pilot2_entities,
    terms
)

relationship_candidates = find_candidates(
    pilot2_relationships,
    terms
)

print("\nENTITY CANDIDATES:")
display(
    entity_candidates[
        ["title", "type", "frequency", "degree", "description"]
    ].head(25)
)

print("\nRELATIONSHIP CANDIDATES:")
display(
    relationship_candidates[
        ["source", "target", "description", "weight"]
    ].head(30)
)


T07: Open-space PV installations should be designed as biodiversity installations in accordance with defined ecological criteria.
Search terms: ['FREIFLÄCHEN', 'BIODIVERSITÄT', 'BIODIVERSITÄTS-PV', 'ÖKOLOGISCH', 'NATURSCHUTZ']

ENTITY CANDIDATES:


,title,type,frequency,degree,description
94,BIODIVERSITÄTS-SOLARPARK,TECHNOLOGY,1,3,A large ground-mounted solar park designed and...
216,ABSCHLÄGE BEI PV-FREIFLÄCHENANLAGEN,SUPPORT_SCHEME,1,3,Financial reductions or deductions for open-sp...
349,BIODIVERSITÄTS-PV-ANLAGE,TECHNOLOGY,1,3,PV systems intentionally designed or managed a...
359,ÖKOLOGISCHER MEHRWERT VON PV-BIODIVERSITÄTSANL...,MARKET_METRIC,1,3,Metric: ecological added value of PV-biodivers...
380,PV-BIODIVERSITÄTSANLAGEN,TECHNOLOGY,1,3,"Photovoltaic biodiversity installations, refer..."
119,KOSTENGÜNSTIGER STROM AUS FREIFLÄCHEN-BIODIVER...,MARKET_METRIC,1,2,Metric: affordability of electricity generated...
220,FREIFLÄCHENANLAGE ALS BIODIVERSITÄTSANLAGE,TECHNOLOGY,1,2,Ground-mounted PV installations designed as bi...
348,FREIFLÄCHENANLAGE,TECHNOLOGY,1,2,"Ground-mounted PV installations (solar parks),..."
360,SOZIALE ZUSTIMMUNG ZU PHOTOVOLTAIK IN ÖSTERREI...,MARKET_METRIC,1,2,Metric: public acceptance of PV projects; Valu...
228,PV-FREIFLÄCHENANLAGE,TECHNOLOGY,1,1,Ground-mounted photovoltaic installations that...



RELATIONSHIP CANDIDATES:


,source,target,description,weight
113,BIODIVERSITÄTS-SOLARPARK,KOSTENGÜNSTIGER STROM AUS FREIFLÄCHEN-BIODIVER...,[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
114,AGRI-PV-ANLAGE,KOSTENGÜNSTIGER STROM AUS FREIFLÄCHEN-BIODIVER...,[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
135,BIODIVERSITÄTS-SOLARPARK,BEVÖLKERUNG,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,8.0
140,BIODIVERSITÄTS-SOLARPARK,KREISLAUFWIRTSCHAFT,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,7.0
252,ABSCHLÄGE BEI PV-FREIFLÄCHENANLAGEN,FOTOVOLTAIKANLAGE,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,9.0
254,ABSCHLÄGE BEI PV-FREIFLÄCHENANLAGEN,ERNEUERBAREN-AUSBAU-GESETZ (EAG),[RELATION_TYPE=IMPLEMENTED_BY] [MODALITY=EXPLI...,10.0
262,FREIFLÄCHENANLAGE ALS BIODIVERSITÄTSANLAGE,FOTOVOLTAIKANLAGE,[RELATION_TYPE=PART_OF] [MODALITY=EXPLICIT_FAC...,10.0
263,FREIFLÄCHENANLAGE ALS BIODIVERSITÄTSANLAGE,RENATURIERUNGSZIELE DER EU,"The entity ""FREIFLÄCHENANLAGE ALS BIODIVERSITÄ...",16.0
272,PV-FREIFLÄCHENANLAGE,ABSCHLÄGE BEI PV-FREIFLÄCHENANLAGEN,[RELATION_TYPE=SUPPORTED_BY] [MODALITY=EXPLICI...,9.0
321,FLÄCHENPOTENTIALE AN FREIFLÄCHEN,INTEGRIERTER ÖSTERREICHISCHER NETZINFRASTRUKTU...,[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,9.0


In [42]:
mask = recall_checklist["fact_id"] == "T07"

recall_checklist.loc[mask, "pilot2_entity_capture"] = "FULL"
recall_checklist.loc[mask, "pilot2_relationship_capture"] = "PARTIAL"
recall_checklist.loc[mask, "pilot2_overall_classification"] = "PARTIALLY_STRUCTURED"

recall_checklist.loc[mask, "pilot2_audit_notes"] = (
    "Pilot 2 extracted open-space PV installations, biodiversity PV installations, "
    "biodiversity and nature-protection concepts. It also connected FREIFLÄCHENANLAGE "
    "with BIODIVERSITÄTS-PV-ANLAGE. However, the requirement to design installations "
    "according to defined ecological criteria was not represented completely as a "
    "structured relationship."
)

recall_checklist.loc[
    mask,
    [
        "fact_id",
        "source_fact",
        "pilot2_entity_capture",
        "pilot2_relationship_capture",
        "pilot2_overall_classification",
        "pilot2_audit_notes",
    ],
]

,fact_id,source_fact,pilot2_entity_capture,pilot2_relationship_capture,pilot2_overall_classification,pilot2_audit_notes
26,T07,Open-space PV installations should be designed...,FULL,PARTIAL,PARTIALLY_STRUCTURED,"Pilot 2 extracted open-space PV installations,..."


In [43]:
fact_id = "T08"

terms = [
    "GEBÄUDE DES BUNDES",
    "BUNDESGEBÄUDE",
    "PV-ANLAGE",
    "TECHNISCH",
    "WIRTSCHAFTLICH",
]

fact = recall_checklist.loc[
    recall_checklist["fact_id"] == fact_id,
    "source_fact"
].iloc[0]

print("\n" + "=" * 90)
print(f"{fact_id}: {fact}")
print("Search terms:", terms)

entity_candidates = find_candidates(pilot2_entities, terms)
relationship_candidates = find_candidates(pilot2_relationships, terms)

print("\nENTITY CANDIDATES:")
display(
    entity_candidates[
        ["title", "type", "frequency", "degree", "description"]
    ].head(25)
)

print("\nRELATIONSHIP CANDIDATES:")
display(
    relationship_candidates[
        ["source", "target", "description", "weight"]
    ].head(30)
)


T08: PV systems should be installed on federal buildings wherever technically and economically possible.
Search terms: ['GEBÄUDE DES BUNDES', 'BUNDESGEBÄUDE', 'PV-ANLAGE', 'TECHNISCH', 'WIRTSCHAFTLICH']

ENTITY CANDIDATES:


,title,type,frequency,degree,description
92,PV-ANLAGE,TECHNOLOGY,5,26,PV-ANLAGE (photovoltaic system) refers to a te...
93,AGRI-PV-ANLAGE,TECHNOLOGY,3,7,AGRI-PV-ANLAGE refers to a form of photovoltai...
42,PV-ANLAGEN,TECHNOLOGY,3,5,PV-ANLAGEN (photovoltaic installations or PV s...
122,PRIORITÄT FÜR DEN EIGENVERBRAUCH AUS PV-ANLAGE...,TARGET,1,4,Target: standard for maximum self-supply from ...
286,INNOVATION IM SOLARTECHNISCHEN BEREICH,TARGET,1,3,Target: increased innovation in solar technolo...
323,UMSATZSTEUERBEFREIUNG FÜR PV-ANLAGEN BIS 35 KW...,SUPPORT_SCHEME,1,3,Zero-percent VAT for PV systems up to 35 kWp i...
327,WIRTSCHAFTLICHKEIT DES PV-ANLAGENBETRIEBS,CONSTRAINT,1,3,"Economic viability of PV operations, stated as..."
332,STEUERLICHE ASPEKTE FÜR PV-ANLAGEN,POLICY,1,3,Policy type: fiscal/tax measure; Jurisdiction:...
349,BIODIVERSITÄTS-PV-ANLAGE,TECHNOLOGY,1,3,PV systems intentionally designed or managed a...
354,PV-ANLAGEN AN LÄRMSCHUTZWÄNDEN,TECHNOLOGY,1,3,PV system deployment on noise barriers in road...



RELATIONSHIP CANDIDATES:


,source,target,description,weight
33,WIRTSCHAFTLICHER BETRIEB VON PHOTOVOLTAIK-ANLAGEN,PHOTOVOLTAIK-ANLAGE,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLI...,10.0
43,PHOTOVOLTAIK-STRATEGIE,TECHNISCHE UND SYSTEMISCHE FRAGEN,[RELATION_TYPE=PROPOSES] [MODALITY=EXPLICIT_FA...,8.0
45,PHOTOVOLTAIK-STRATEGIE,WIRTSCHAFTLICHKEIT DER PV-ANLAGEN,[RELATION_TYPE=PROPOSES] [MODALITY=EXPLICIT_FA...,8.0
51,PV-ANLAGEN,"PV-GENERATION — 6.3 TWH (ÖSTERREICH, 2023)",[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
60,ENERGIEINFRASTRUKTUR,TECHNISCHE UND SYSTEMISCHE FRAGEN,[RELATION_TYPE=ASSOCIATED_WITH] [MODALITY=EXPL...,8.0
89,PV-ANLAGEN,GEBÄUDE,[RELATION_TYPE=LOCATED_IN] [MODALITY=EXPLICIT_...,9.0
90,PV-ANLAGEN,STROMERZEUGUNGSFUNKTION ALLER NEUEN UND GRUNDS...,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=SCENA...,9.0
105,ELEKTROTECHNISCHE NORMUNGSORGANISATION,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,[RELATION_TYPE=RESPONSIBLE_FOR] [MODALITY=EXPL...,8.0
113,BIODIVERSITÄTS-SOLARPARK,KOSTENGÜNSTIGER STROM AUS FREIFLÄCHEN-BIODIVER...,[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0
114,AGRI-PV-ANLAGE,KOSTENGÜNSTIGER STROM AUS FREIFLÄCHEN-BIODIVER...,[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0


In [44]:
mask = recall_checklist["fact_id"] == "T08"

recall_checklist.loc[mask, "pilot2_entity_capture"] = "PARTIAL"
recall_checklist.loc[mask, "pilot2_relationship_capture"] = "MISSING"
recall_checklist.loc[mask, "pilot2_overall_classification"] = "ENTITY_ONLY"

recall_checklist.loc[mask, "pilot2_audit_notes"] = (
    "Pilot 2 extracted general PV-system, building, technical, and economic "
    "concepts, but it did not extract federal buildings as a distinct entity. "
    "It also did not represent the complete relationship stating that PV "
    "systems should be installed on federal buildings wherever technically "
    "and economically possible."
)

recall_checklist.loc[
    mask,
    [
        "fact_id",
        "source_fact",
        "pilot2_entity_capture",
        "pilot2_relationship_capture",
        "pilot2_overall_classification",
        "pilot2_audit_notes",
    ],
]

,fact_id,source_fact,pilot2_entity_capture,pilot2_relationship_capture,pilot2_overall_classification,pilot2_audit_notes
27,T08,PV systems should be installed on federal buil...,PARTIAL,MISSING,ENTITY_ONLY,"Pilot 2 extracted general PV-system, building,..."


In [45]:
fact_id = "T09"

terms = [
    "ENERGIEGEMEINSCHAFTEN",
    "SPEICHER",
    "FLEXIBILITÄT",
    "ECHTZEIT",
    "VERBRAUCH",
]

fact = recall_checklist.loc[
    recall_checklist["fact_id"] == fact_id,
    "source_fact",
].iloc[0]

print("\n" + "=" * 90)
print(f"{fact_id}: {fact}")
print("Search terms:", terms)

entity_candidates = find_candidates(
    pilot2_entities,
    terms,
)

relationship_candidates = find_candidates(
    pilot2_relationships,
    terms,
)

print("\nENTITY CANDIDATES:")
display(
    entity_candidates[
        [
            "title",
            "type",
            "frequency",
            "degree",
            "description",
        ]
    ].head(20)
)

print("\nRELATIONSHIP CANDIDATES:")
display(
    relationship_candidates[
        [
            "source",
            "target",
            "description",
            "weight",
        ]
    ].head(25)
)


T09: Energy communities should be further developed using storage, consumption flexibility, real-time monitoring and control.
Search terms: ['ENERGIEGEMEINSCHAFTEN', 'SPEICHER', 'FLEXIBILITÄT', 'ECHTZEIT', 'VERBRAUCH']

ENTITY CANDIDATES:


,title,type,frequency,degree,description
111,ENERGIEGEMEINSCHAFTEN,STAKEHOLDER,3,11,ENERGIEGEMEINSCHAFTEN refers to energy communi...
478,ENERGIEGEMEINSCHAFTEN (EGS),STAKEHOLDER,1,6,Energy Communities (EGs) are collective models...
8,100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,TARGET,1,5,Target: nationally balanced renewable electric...
122,PRIORITÄT FÜR DEN EIGENVERBRAUCH AUS PV-ANLAGE...,TARGET,1,4,Target: standard for maximum self-supply from ...
227,100% NATIONAL BILANZIELL ERNEUERBARER GESAMTST...,TARGET,1,4,Target: complete national coverage of total el...
308,GROSSSPEICHER,INFRASTRUCTURE,1,4,Large-scale storage systems for long-term elec...
170,STROM- UND FLEXIBILITÄTSMÄRKTE,INFRASTRUCTURE,2,3,STROM- UND FLEXIBILITÄTSMÄRKTE are electricity...
292,FLEXIBILITÄTSSTUDIEN DER E-CONTROL,POLICY,1,3,Policy type: studies/research; Jurisdiction: A...
72,ENERGIEVERBRAUCHSREDUKTION,TARGET,1,2,Target: reduction of energy consumption; Targe...
97,MOBILE UND STATIONÄRE SPEICHER,TECHNOLOGY,1,2,Mobile and stationary storage units that are i...



RELATIONSHIP CANDIDATES:


,source,target,description,weight
4,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLI...,9.0
5,PHOTOVOLTAIK,100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLI...,10.0
7,ÖSTERREICH,100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,[RELATION_TYPE=HAS_TARGET] [MODALITY=EXPLICIT_...,10.0
8,ERNEUERBAREN-AUSBAU-GESETZ (EAG),100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,[RELATION_TYPE=SETS_TARGET] [MODALITY=EXPLICIT...,10.0
15,ERNEUERBARE ENERGIEN,100% BILANZIELL ERNEUERBARER STROMVERBRAUCH BI...,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLI...,10.0
38,ERNEUERBAREN-AUSBAU-GESETZ (EAG),100% NATIONAL BILANZIERTER ERNEUERBARER STROMV...,[RELATION_TYPE=SETS_TARGET] [MODALITY=EXPLICIT...,10.0
57,EU-ERNEUERBAREN-RICHTLINIE (RED III),ANTEIL ERNEUERBARER ENERGIEN AM GESAMTENERGIEV...,[RELATION_TYPE=SETS_TARGET] [MODALITY=EXPLICIT...,10.0
92,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,ENERGIEVERBRAUCHSREDUKTION,[RELATION_TYPE=SETS_TARGET] [MODALITY=PLANNED_...,8.0
100,ENERGIEEFFIZIENZ,ENERGIEVERBRAUCHSREDUKTION,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLI...,8.0
119,MOBILE UND STATIONÄRE SPEICHER,PV-ANLAGE,[RELATION_TYPE=PART_OF] [MODALITY=EXPLICIT_FAC...,10.0


In [46]:
mask = recall_checklist["fact_id"] == "T09"

recall_checklist.loc[mask, "pilot2_entity_capture"] = "FULL"
recall_checklist.loc[mask, "pilot2_relationship_capture"] = "PARTIAL"
recall_checklist.loc[mask, "pilot2_overall_classification"] = "PARTIALLY_STRUCTURED"

recall_checklist.loc[mask, "pilot2_audit_notes"] = (
    "Pilot 2 extracted energy communities, storage, flexible consumers, "
    "flexibility incentives and related concepts. Several relevant relationships "
    "were extracted, but the complete source statement connecting energy communities "
    "with storage, consumption flexibility and real-time mechanisms was not represented "
    "as one explicit structured relationship."
)

recall_checklist.loc[
    mask,
    [
        "fact_id",
        "source_fact",
        "pilot2_entity_capture",
        "pilot2_relationship_capture",
        "pilot2_overall_classification",
        "pilot2_audit_notes",
    ],
]

,fact_id,source_fact,pilot2_entity_capture,pilot2_relationship_capture,pilot2_overall_classification,pilot2_audit_notes
28,T09,Energy communities should be further developed...,FULL,PARTIAL,PARTIALLY_STRUCTURED,"Pilot 2 extracted energy communities, storage,..."


In [47]:
fact_id = "T10"

terms = [
    "FORSCHUNG",
    "RESSOURCENEFFIZIENZ",
    "LANGLEBIGKEIT",
    "REPARATUR",
    "WIEDERVERWENDUNG",
    "RECYCLING",
    "KREISLAUFWIRTSCHAFT",
]

fact = recall_checklist.loc[
    recall_checklist["fact_id"] == fact_id,
    "source_fact",
].iloc[0]

print("\n" + "=" * 90)
print(f"{fact_id}: {fact}")
print("Search terms:", terms)

entity_candidates = find_candidates(pilot2_entities, terms)
relationship_candidates = find_candidates(pilot2_relationships, terms)

print("\nENTITY CANDIDATES:")
display(
    entity_candidates[
        ["title", "type", "frequency", "degree", "description"]
    ].head(20)
)

print("\nRELATIONSHIP CANDIDATES:")
display(
    relationship_candidates[
        ["source", "target", "description", "weight"]
    ].head(25)
)


T10: PV research should address resource efficiency, durability, repairability, reuse and recycling of PV components.
Search terms: ['FORSCHUNG', 'RESSOURCENEFFIZIENZ', 'LANGLEBIGKEIT', 'REPARATUR', 'WIEDERVERWENDUNG', 'RECYCLING', 'KREISLAUFWIRTSCHAFT']

ENTITY CANDIDATES:


,title,type,frequency,degree,description
126,KREISLAUFWIRTSCHAFT,SUPPORT_SCHEME,1,4,The circular-economy approach is applied at al...
421,PV-KOMPONENTEN,TECHNOLOGY,2,2,PV-KOMPONENTEN refers to photovoltaic (PV) sys...
172,FORSCHUNGSEINRICHTUNGEN,ORGANIZATION,1,2,Leading research institutions in Austria are m...
186,FORSCHUNGSEINRICHTUNGEN IM BEREICH DER PHOTOVO...,ORGANIZATION,1,2,Research institutions in the field of photovol...
189,FORSCHUNG-TECHNOLOGIE-INNOVATION,POLICY,1,2,Policy type: action field/strategy area; Juris...
272,"PRODUKTQUALITÄT, INSTALLATION, SERVICE, REPARA...",SUPPORT_SCHEME,1,2,Support scheme involving measures to assure qu...
412,FORSCHUNG,STAKEHOLDER,1,2,Research institutions and actors involved in P...
461,FTI-SCHWERPUNKT KREISLAUFWIRTSCHAFT & PRODUKTION,POLICY,1,2,Policy type: national research and innovation ...
57,FORSCHUNG-TECHNOLOGIE-INNOVATION (FTI),POLICY,1,1,Policy type: strategy/action field; Jurisdicti...
88,EUROPÄISCHES FORSCHUNGSRAHMENPROGRAMM,POLICY,1,1,Policy type: programme; Jurisdiction: EU; Stat...



RELATIONSHIP CANDIDATES:


,source,target,description,weight
47,PHOTOVOLTAIK-STRATEGIE,FORSCHUNG-TECHNOLOGIE-INNOVATION (FTI),[RELATION_TYPE=PROPOSES] [MODALITY=EXPLICIT_FA...,8.0
97,EUROPÄISCHES FORSCHUNGSRAHMENPROGRAMM,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,[RELATION_TYPE=ASSOCIATED_WITH] [MODALITY=RECO...,6.0
121,KREISLAUFWIRTSCHAFT,HOHE PRODUKTQUALITÄT PV-SYSTEME,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,9.0
139,KREISLAUFWIRTSCHAFT,PV-ANLAGE,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,8.0
140,BIODIVERSITÄTS-SOLARPARK,KREISLAUFWIRTSCHAFT,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,7.0
152,RECYCLINGFÄHIGKEIT PV-PRODUKTE,KREISLAUFWIRTSCHAFT,[RELATION_TYPE=ASSOCIATED_WITH] [MODALITY=EXPL...,9.0
197,FORSCHUNGSEINRICHTUNGEN,ÖSTERREICH,[RELATION_TYPE=LOCATED_IN] [MODALITY=EXPLICIT_...,8.0
210,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,FORSCHUNG-TECHNOLOGIE-INNOVATION,[RELATION_TYPE=PROPOSES] [MODALITY=EXPLICIT_FA...,10.0
218,FORSCHUNGSEINRICHTUNGEN IM BEREICH DER PHOTOVO...,FORSCHUNG-TECHNOLOGIE-INNOVATION,[RELATION_TYPE=ASSOCIATED_WITH] [MODALITY=EXPL...,10.0
236,FÜHRENDE FORSCHUNGSEINRICHTUNGEN IM BEREICH DE...,FORSCHUNGSEINRICHTUNGEN IM BEREICH DER PHOTOVO...,[RELATION_TYPE=ALIAS_OF] [MODALITY=EXPLICIT_FA...,10.0


In [48]:
mask = recall_checklist["fact_id"] == "T10"

recall_checklist.loc[mask, "pilot2_entity_capture"] = "FULL"
recall_checklist.loc[mask, "pilot2_relationship_capture"] = "PARTIAL"
recall_checklist.loc[
    mask,
    "pilot2_overall_classification"
] = "PARTIALLY_STRUCTURED"

recall_checklist.loc[mask, "pilot2_audit_notes"] = (
    "Pilot 2 extracted research and innovation, circular economy, "
    "resource efficiency, durability, repair, recycling and related "
    "PV concepts. Several supporting relationships were generated, "
    "but the complete source statement was distributed across multiple "
    "entities and relationships rather than represented as one precise "
    "relationship connecting PV research to all stated objectives."
)

recall_checklist.loc[
    mask,
    [
        "fact_id",
        "source_fact",
        "pilot2_entity_capture",
        "pilot2_relationship_capture",
        "pilot2_overall_classification",
        "pilot2_audit_notes",
    ],
]

,fact_id,source_fact,pilot2_entity_capture,pilot2_relationship_capture,pilot2_overall_classification,pilot2_audit_notes
29,T10,PV research should address resource efficiency...,FULL,PARTIAL,PARTIALLY_STRUCTURED,"Pilot 2 extracted research and innovation, cir..."


In [49]:
print("PILOT 2 RECALL AUDIT — FINAL RESULTS")
print("=" * 50)

print("\nTotal source facts reviewed:")
print(len(recall_checklist))

print("\nEntity capture:")
print(recall_checklist["pilot2_entity_capture"].value_counts())

print("\nEntity capture percentages:")
print(
    recall_checklist["pilot2_entity_capture"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

print("\nRelationship capture:")
print(recall_checklist["pilot2_relationship_capture"].value_counts())

print("\nRelationship capture percentages:")
print(
    recall_checklist["pilot2_relationship_capture"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

print("\nOverall fact-level classification:")
print(recall_checklist["pilot2_overall_classification"].value_counts())

print("\nOverall classification percentages:")
print(
    recall_checklist["pilot2_overall_classification"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

print("\nResults by source-fact category:")
print(
    pd.crosstab(
        recall_checklist["category"],
        recall_checklist["pilot2_overall_classification"],
    )
)

PILOT 2 RECALL AUDIT — FINAL RESULTS

Total source facts reviewed:
30

Entity capture:
pilot2_entity_capture
FULL       20
PARTIAL     5
            5
Name: count, dtype: int64

Entity capture percentages:
pilot2_entity_capture
FULL       66.7
PARTIAL    16.7
           16.7
Name: proportion, dtype: float64

Relationship capture:
pilot2_relationship_capture
PARTIAL    11
FULL        9
MISSING     5
            5
Name: count, dtype: int64

Relationship capture percentages:
pilot2_relationship_capture
PARTIAL    36.7
FULL       30.0
MISSING    16.7
           16.7
Name: proportion, dtype: float64

Overall fact-level classification:
pilot2_overall_classification
PARTIALLY_STRUCTURED    11
FULLY_STRUCTURED         9
ENTITY_ONLY              5
                         5
Name: count, dtype: int64

Overall classification percentages:
pilot2_overall_classification
PARTIALLY_STRUCTURED    36.7
FULLY_STRUCTURED        30.0
ENTITY_ONLY             16.7
                        16.7
Name: proportio

In [50]:
output_path = (
    project_root
    / "pilot_02"
    / "inspection"
    / "14_recall_coverage_checklist_completed.csv"
)

recall_checklist.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
)

print("Completed Pilot 2 recall checklist saved to:")
print(output_path)

Completed Pilot 2 recall checklist saved to:
C:\Users\HP-ZBOOK i7\graphrag_test\pilot_02\inspection\14_recall_coverage_checklist_completed.csv


In [51]:
market_facts_completion = {
    "M01": {
        "entity": "FULL",
        "relationship": "FULL",
        "overall": "FULLY_STRUCTURED",
        "notes": (
            "Pilot 2 extracted the 6.3 TWh PV-generation metric, Austria, "
            "Photovoltaics and the year 2023, with a relationship representing "
            "the measured PV-generation result."
        ),
    },
    "M02": {
        "entity": "FULL",
        "relationship": "FULL",
        "overall": "FULLY_STRUCTURED",
        "notes": (
            "Pilot 2 extracted the annual PV-capacity addition of approximately "
            "2.5 GW in Austria in 2023 and represented it through an explicit "
            "measured relationship."
        ),
    },
    "M03": {
        "entity": "FULL",
        "relationship": "FULL",
        "overall": "FULLY_STRUCTURED",
        "notes": (
            "Pilot 2 extracted the annual PV-capacity addition of 1 GW in "
            "Austria in 2022 and represented the numerical result through an "
            "explicit measured relationship."
        ),
    },
    "M04": {
        "entity": "FULL",
        "relationship": "PARTIAL",
        "overall": "PARTIALLY_STRUCTURED",
        "notes": (
            "Pilot 2 extracted the Integrated Austrian Network Infrastructure "
            "Plan, Photovoltaics and the 41 TWh by 2040 target. However, the "
            "meaning is distributed across relationships rather than represented "
            "as one direct NIP-to-target relationship."
        ),
    },
    "M05": {
        "entity": "FULL",
        "relationship": "FULL",
        "overall": "FULLY_STRUCTURED",
        "notes": (
            "Pilot 2 extracted the building-based PV potential of approximately "
            "15 TWh by 2040 and connected it to the broader 41 TWh photovoltaic "
            "potential."
        ),
    },
}

for fact_id, result in market_facts_completion.items():
    mask = recall_checklist["fact_id"] == fact_id

    recall_checklist.loc[mask, "pilot2_entity_capture"] = result["entity"]
    recall_checklist.loc[mask, "pilot2_relationship_capture"] = result["relationship"]
    recall_checklist.loc[mask, "pilot2_overall_classification"] = result["overall"]
    recall_checklist.loc[mask, "pilot2_audit_notes"] = result["notes"]

recall_checklist.loc[
    recall_checklist["fact_id"].isin(["M01", "M02", "M03", "M04", "M05"]),
    [
        "fact_id",
        "source_fact",
        "pilot2_entity_capture",
        "pilot2_relationship_capture",
        "pilot2_overall_classification",
        "pilot2_audit_notes",
    ],
]

,fact_id,source_fact,pilot2_entity_capture,pilot2_relationship_capture,pilot2_overall_classification,pilot2_audit_notes
10,M01,Austria reached 6.3 TWh of PV generation in 2023.,FULL,FULL,FULLY_STRUCTURED,Pilot 2 extracted the 6.3 TWh PV-generation me...
11,M02,Approximately 2.5 GW of new photovoltaic capac...,FULL,FULL,FULLY_STRUCTURED,Pilot 2 extracted the annual PV-capacity addit...
12,M03,One GW of new photovoltaic capacity was instal...,FULL,FULL,FULLY_STRUCTURED,Pilot 2 extracted the annual PV-capacity addit...
13,M04,The Integrated Austrian Network Infrastructure...,FULL,PARTIAL,PARTIALLY_STRUCTURED,Pilot 2 extracted the Integrated Austrian Netw...
14,M05,At least approximately 15 TWh of the 2040 PV p...,FULL,FULL,FULLY_STRUCTURED,Pilot 2 extracted the building-based PV potent...


In [52]:
required_columns = [
    "pilot2_entity_capture",
    "pilot2_relationship_capture",
    "pilot2_overall_classification",
    "pilot2_audit_notes",
]

print("Missing values:")
print(recall_checklist[required_columns].isna().sum())

Missing values:
pilot2_entity_capture            0
pilot2_relationship_capture      0
pilot2_overall_classification    0
pilot2_audit_notes               0
dtype: int64


In [53]:
print("PILOT 2 RECALL AUDIT — FINAL RESULTS")
print("=" * 50)

print("\nTotal source facts reviewed:")
print(len(recall_checklist))

print("\nEntity capture:")
print(recall_checklist["pilot2_entity_capture"].value_counts())

print("\nEntity capture percentages:")
print(
    recall_checklist["pilot2_entity_capture"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

print("\nRelationship capture:")
print(recall_checklist["pilot2_relationship_capture"].value_counts())

print("\nRelationship capture percentages:")
print(
    recall_checklist["pilot2_relationship_capture"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

print("\nOverall fact-level classification:")
print(recall_checklist["pilot2_overall_classification"].value_counts())

print("\nOverall classification percentages:")
print(
    recall_checklist["pilot2_overall_classification"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

print("\nOverall classifications by category:")
print(
    pd.crosstab(
        recall_checklist["category"],
        recall_checklist["pilot2_overall_classification"],
    )
)

PILOT 2 RECALL AUDIT — FINAL RESULTS

Total source facts reviewed:
30

Entity capture:
pilot2_entity_capture
FULL       25
PARTIAL     5
Name: count, dtype: int64

Entity capture percentages:
pilot2_entity_capture
FULL       83.3
PARTIAL    16.7
Name: proportion, dtype: float64

Relationship capture:
pilot2_relationship_capture
FULL       13
PARTIAL    12
MISSING     5
Name: count, dtype: int64

Relationship capture percentages:
pilot2_relationship_capture
FULL       43.3
PARTIAL    40.0
MISSING    16.7
Name: proportion, dtype: float64

Overall fact-level classification:
pilot2_overall_classification
FULLY_STRUCTURED        13
PARTIALLY_STRUCTURED    12
ENTITY_ONLY              5
Name: count, dtype: int64

Overall classification percentages:
pilot2_overall_classification
FULLY_STRUCTURED        43.3
PARTIALLY_STRUCTURED    40.0
ENTITY_ONLY             16.7
Name: proportion, dtype: float64

Overall classifications by category:
pilot2_overall_classification  ENTITY_ONLY  FULLY_STRUCTURED

In [54]:
output_path = (
    project_root
    / "pilot_02"
    / "inspection"
    / "14_recall_coverage_checklist_completed.csv"
)

recall_checklist.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
)

print("Saved to:")
print(output_path)

Saved to:
C:\Users\HP-ZBOOK i7\graphrag_test\pilot_02\inspection\14_recall_coverage_checklist_completed.csv


# Pilot 2 Recall and Source-Fact Coverage Report

## 1. Purpose

This inspection evaluates whether Microsoft GraphRAG Pilot 2 successfully transformed important facts from the Austrian Photovoltaic Strategy into structured graph information.

Recall was evaluated using a manually prepared checklist of 30 representative source facts:

- 10 policy facts
- 10 market and numerical facts
- 10 technical facts

For each source fact, the inspection examined four stages:

1. Whether the fact remained available in the GraphRAG text units
2. Whether its important entities were extracted
3. Whether its meaning was represented through relationships
4. Whether the complete fact became a structured graph statement

The evaluation compares Pilot 2 with the earlier Pilot 1 results.

---

## 2. Evaluation categories

### Text-unit capture

- `YES`: The relevant source statement is present in at least one GraphRAG text unit.
- `PARTIAL`: Only part of the statement or its necessary context is present.
- `MISSING`: The statement is not preserved in the text units.

### Entity capture

- `FULL`: All central concepts required to represent the fact were extracted as entities.
- `PARTIAL`: Some central concepts were extracted, but at least one important concept, quantity, date, unit, actor or condition was missing.
- `MISSING`: The central concepts were not represented as entities.

### Relationship capture

- `FULL`: The graph contains a relationship that substantially preserves the source fact.
- `PARTIAL`: The graph represents part of the meaning, or distributes it across several less precise relationships.
- `MISSING`: The relevant entities may exist, but the source fact is not represented through an adequate relationship.

### Overall fact-level classification

- `FULLY_STRUCTURED`: The important entities and their relationship are represented adequately.
- `PARTIALLY_STRUCTURED`: The graph contains the principal information, but the complete meaning is fragmented, incomplete or insufficiently precise.
- `ENTITY_ONLY`: Relevant entities exist, but no adequate relationship represents the fact.
- `MISSING_FROM_GRAPH`: The source fact is not meaningfully represented in the graph.
- `RELATIONSHIP_TEXT_BUT_ENTITY_INCOMPLETE`: Relationship text contains relevant meaning, but the entity representation is incomplete.

---

## 3. Text-unit coverage

All 30 selected source facts were found in the Pilot 2 text units.

| Text-unit capture | Count | Percentage |
|---|---:|---:|
| Yes | 30 | 100.0% |
| Partial | 0 | 0.0% |
| Missing | 0 | 0.0% |

This confirms that the chunking stage did not remove the selected policy, market/numerical or technical information.

Pilot 1 and Pilot 2 used the same converted document text and the same 17 text units. Therefore, differences in graph recall cannot be attributed to document conversion or chunking. They result principally from changes to the entity schema and extraction prompts.

---

## 4. Entity recall in Pilot 2

| Entity capture | Count | Percentage |
|---|---:|---:|
| Full | 25 | 83.3% |
| Partial | 5 | 16.7% |
| Missing | 0 | 0.0% |

Every selected source fact received at least partial entity representation. No evaluated fact was completely absent at the entity level.

### Entity capture by category

| Category | Full | Partial | Missing |
|---|---:|---:|---:|
| Policy | 8 | 2 | 0 |
| Market/numerical | 10 | 0 | 0 |
| Technical | 7 | 3 | 0 |

The strongest result occurred for market and numerical information: all 10 selected facts had full entity capture. This is an important improvement because Pilot 2 introduced controlled classes such as `TARGET` and `MARKET_METRIC`.

The remaining partial cases primarily concern source statements containing several technical concepts, conditions or actors that could not be represented completely by one extracted entity set.

---

## 5. Relationship recall in Pilot 2

| Relationship capture | Count | Percentage |
|---|---:|---:|
| Full | 13 | 43.3% |
| Partial | 12 | 40.0% |
| Missing | 5 | 16.7% |

Twenty-five of the 30 facts received at least partial relationship representation. However, only 13 were represented completely.

### Relationship capture by category

| Category | Full | Partial | Missing |
|---|---:|---:|---:|
| Policy | 5 | 4 | 1 |
| Market/numerical | 7 | 2 | 1 |
| Technical | 1 | 6 | 3 |

Market and numerical relationships performed best. Seven of the 10 numerical facts were fully represented. Examples include:

- 6.3 TWh of PV generation in 2023
- 2.5 GW of additional PV capacity in 2023
- 1 GW of additional PV capacity in 2022
- 19 TWh growth relative to 2020
- approximately EUR 600 million in PV support
- the approximate PV shares of Austrian electricity and total energy demand

Technical relationships remained considerably more difficult. Only one of the 10 technical facts was fully structured. Six were partially represented, and three lacked an adequate relationship.

This indicates that GraphRAG can identify technical concepts more reliably than it can preserve complex technical propositions connecting technologies, infrastructure, operational mechanisms, conditions and expected effects.

---

## 6. Overall fact-level recall

| Overall classification | Count | Percentage |
|---|---:|---:|
| Fully structured | 13 | 43.3% |
| Partially structured | 12 | 40.0% |
| Entity only | 5 | 16.7% |
| Missing from graph | 0 | 0.0% |

A total of 25 of 30 facts, or 83.3%, were at least partially structured. Thirteen facts, or 43.3%, were fully structured.

No selected source fact was completely missing from the Pilot 2 graph.

### Overall result by category

| Category | Fully structured | Partially structured | Entity only |
|---|---:|---:|---:|
| Policy | 5 | 4 | 1 |
| Market/numerical | 7 | 2 | 1 |
| Technical | 1 | 6 | 3 |

The controlled Pilot 2 configuration was particularly effective for market and numerical information. Technical facts remain the principal source of incomplete graph representation.

---

## 7. Comparison with Pilot 1

### Overall fact-level comparison

| Classification | Pilot 1 | Pilot 1 percentage | Pilot 2 | Pilot 2 percentage |
|---|---:|---:|---:|---:|
| Fully structured | 1 | 3.3% | 13 | 43.3% |
| Partially structured | 13 | 43.3% | 12 | 40.0% |
| Entity only | 14 | 46.7% | 5 | 16.7% |
| Missing from graph | 1 | 3.3% | 0 | 0.0% |
| Relationship text but entity incomplete | 1 | 3.3% | 0 | 0.0% |

The number of fully structured facts increased from 1 in Pilot 1 to 13 in Pilot 2.

This represents:

- an absolute improvement of 12 fully structured facts;
- an increase from 3.3% to 43.3%;
- a reduction in entity-only facts from 14 to 5;
- elimination of completely missing facts in the evaluated sample.

The results provide strong evidence that the controlled entity schema and revised extraction prompt improved the graph’s ability to preserve domain-relevant source facts.

---

## 8. Main improvements produced by Pilot 2

### 8.1 Numerical information became graph content

Pilot 1 often mentioned numerical information only inside entity or relationship descriptions. Pilot 2 more frequently created dedicated `TARGET` and `MARKET_METRIC` entities and connected them to policies, technologies, geographic areas or time periods.

This substantially improved the representation of:

- measured PV generation;
- annual capacity additions;
- future generation targets;
- deadlines;
- percentages;
- financial support amounts.

### 8.2 Entity identification improved

No selected fact was completely missing at the entity level. The expanded schema allowed GraphRAG to distinguish policies, technologies, infrastructure, targets, constraints, support schemes and market metrics instead of forcing most concepts into generic categories.

### 8.3 Policy representation improved

Five of the 10 selected policy facts were fully structured, while another four were partially structured. Relevant laws, plans, strategies, targets and support mechanisms were represented more clearly than in Pilot 1.

### 8.4 Modality became more explicit

Pilot 2 relationships included markers such as:

- `EXPLICIT_FACT`
- `PLANNED_ACTION`
- `RECOMMENDATION`
- `SCENARIO`
- `PROPOSAL`

These markers help distinguish observed results from plans, recommendations and scenarios. This is important for avoiding the interpretation of proposed measures as completed actions.

---

## 9. Remaining recall limitations

### 9.1 Relationship construction remains the main bottleneck

Entity recall reached 83.3% full capture, while relationship recall reached only 43.3% full capture.

This difference shows that GraphRAG is more successful at recognizing important concepts than at converting complete source propositions into precise graph relationships.

### 9.2 Complex facts are fragmented

Some source facts were represented across several entities and relationships instead of one coherent statement.

For example, GraphRAG may extract:

- a policy;
- a numerical target;
- a deadline;
- a technology;
- and a geographic area;

but fail to connect all of them through one precise relationship structure.

### 9.3 Technical propositions remain difficult

The weakest category was technical information:

- 1 fully structured fact;
- 6 partially structured facts;
- 3 entity-only facts.

Technical statements often combine several components, such as a technology, infrastructure element, operational problem, mitigation mechanism and expected outcome. A simple binary relationship is often insufficient to preserve the complete meaning.

### 9.4 Some important relationships remain absent

The five entity-only cases demonstrate that extracting the required concepts does not guarantee extraction of the source proposition.

Examples include cases where:

- a policy is extracted but its evolving status is not represented;
- an annual expansion target exists without a relationship to the long-term potential;
- technologies are extracted without their stated integration relationship;
- technical orientation concepts are extracted without their expected production effect;
- buildings and PV systems exist without the complete feasibility condition.

### 9.5 Partial recall is not equivalent to correctness

This recall analysis measures whether important source information appears in the graph. It does not establish that every extracted relationship is correct.

The separate relationship-precision audit found that some relationships had:

- incorrect direction or endpoints;
- reasonable but non-explicit inferences;
- overgeneralization;
- unsupported statements.

Recall and precision must therefore be interpreted together.

---

## 10. Interpretation

Pilot 2 is a substantial improvement over Pilot 1.

The controlled schema and revised prompts successfully changed GraphRAG from a graph dominated by generic entity categories into a more useful solar-market extraction layer. Important policies, technologies, infrastructure elements, targets, constraints, support schemes and market metrics are now much more visible.

However, Pilot 2 is not yet a fully reliable final knowledge graph. It should be treated as a domain-aware but noisy extraction layer.

The results indicate the following pipeline:

**Source text → strong text preservation → strong entity recall → moderate relationship recall → controlled downstream correction required**

The main remaining task is no longer basic entity discovery. It is the normalization and validation of structured relationships.

---

## 11. Recommended next steps

1. Preserve the Pilot 2 controlled entity schema.
2. Retain explicit modality markers for facts, plans, scenarios and recommendations.
3. Add controlled downstream entity resolution for abbreviations, spelling variants and aliases.
4. Validate relationship direction and endpoint types.
5. Normalize extracted relationship labels into a smaller controlled vocabulary.
6. Represent quantitative statements using structured fields for value, unit, year, status and geographic scope.
7. Consider statement or event nodes for complex facts that cannot be represented adequately by one binary edge.
8. Retain provenance linking every controlled fact to its source text unit and document.
9. Use GraphRAG as an extraction and retrieval layer, followed by controlled KG transformation and validation.
10. Test the revised method on additional documents before fixing the final ontology.

---

## 12. Final conclusion

Pilot 2 preserved all 30 selected source facts in its text units and provided at least partial entity representation for every fact.

Full entity capture reached 83.3%, while full relationship capture reached 43.3%. Overall, 43.3% of facts were fully structured and another 40.0% were partially structured.

Compared with Pilot 1, the proportion of fully structured facts increased from 3.3% to 43.3%. This confirms that the controlled domain schema and revised prompts produced a meaningful improvement.

Nevertheless, relationship construction remains the main recall bottleneck, particularly for complex technical statements. The recommended architecture is therefore to retain GraphRAG as a domain-aware extraction layer while applying controlled entity resolution, relationship normalization, provenance management and validation downstream.